# **Simuladores: Sistemas de Retroalimentación**
### Capítulo 7 — *Aprendizaje y Comportamiento Adaptable: Principios y Modelos*
**Arturo Bouzas** · Facultad de Psicología, UNAM · bouzaslab25.com

---

Este notebook contiene tres simuladores del Capítulo 5:

| Simulador | Concepto central |
|-----------|-----------------|
| **1 — Fototaxia básica** | Comparación simultánea bilateral, ganancia, taxia positiva/negativa |
| **2 — Efecto de la demora** | Inestabilidad en sistemas de retroalimentación con latencia |
| **3 — Reactivo vs. Anticipatorio** | Límites de la retroalimentación pura; necesidad de predicción |

Corre las celdas en orden o ejecuta todas previo a la manipulación del simulador. Se requieren las importaciones contenidas en la celda del primera simulador.

---
## **Simulador 1 — Fototaxia Básica**

Una polilla con dos sensores laterales de intensidad luminosa navega hacia (o lejos de) una fuente puntual.

**Ecuación de control:**
$$\omega = k \cdot (S_{der} - S_{izq})$$

donde $\omega$ es la tasa de giro, $k$ es la ganancia del sistema, y $S_{der} - S_{izq}$ es la señal de error.



In [ ]:
#@title ─── *Celda 1: Visualización* ──────────────────────────────────
# ============================================================
# CELDA 1 — Lógica del agente y visualización estática
# Fototaxia positiva · Retroalimentación negativa
# Tropotaxia: comparación bilateral simultánea
# ============================================================

# ========= IMPORTACIONES =========
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import matplotlib.gridspec as gridspec
from collections import deque
import ipywidgets as widgets
from IPython.display import display, clear_output
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.collections import LineCollection
import warnings
warnings.filterwarnings("ignore")
from matplotlib.patches import Patch
from matplotlib.gridspec import GridSpec

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

# ── Paletas de colores ───────────────────────────────────────
PALETTES = {
    'dark': dict(
        DARK_BG  = "#0d1117",
        PANEL_BG = "#161b22",
        ACCENT1  = "#ff7832",   # naranja — error positivo
        ACCENT2  = "#7ed6f8",   # azul    — error negativo
        GOLD     = "#f0c040",   # dorado  — variable principal
        TEXT     = "#c9d1d9",
        SUBTEXT  = "#8b949e",
        LINE     = "#30363d",
        MIDTONE  = "#1a1f2e",   # color central del cmap de error
        GREEN    = "#3fb950",   # verde (convergencia)
        RED      = "#f85149",   # rojo  (divergencia)
        PURPLE   ='#bc8cff',
    ),

    'light': dict(
        DARK_BG  = "#f6f8fa",
        PANEL_BG = "#ffffff",
        ACCENT1  = "#c4390a",   # Rojo — error positivo
        ACCENT2  = "#0969da",   # azul — error negativo
        GOLD     = "#f0c040",   # dorado  — variable principal
        TEXT     = "#1f2328",
        SUBTEXT  = "#57606a",
        LINE     = "#d0d7de",
        MIDTONE  = "#e8ecf0",   # color central del cmap de error
        GREEN    = "#3fb950",   # verde (convergencia)
        RED      = "#f85149",   # rojo  (divergencia)
        PURPLE   ='#bc8cff',
    ),
}

P = dict(PALETTES['dark'])

# ── Parámetros de la simulación ──────────────────────────────
GAIN_K      = 0.15    # ganancia k (k > 0 → fototaxia positiva)
V_FORWARD   = 1       # velocidad de avance (u.a. / paso), constante
N_STEPS     = 90      # pasos de simulación
THETA0_DEG  = 150.0   # ángulo relativo inicial θ₀ (grados)
DT          = 1.0     # tamaño del paso temporal
NOISE_SIGMA = 0.0     # ruido ambiental

LIGHT_POS   = np.array([ 55,  45])    # posición de la fuente de luz
AGENT_START = np.array([0, 0])        # posición inicial del agente

# ── Funciones del sistema ────────────────────────────────────

def normalize_angle(a):
    """Normaliza un ángulo al intervalo (−π, π]."""
    return (a + np.pi) % (2 * np.pi) - np.pi


def compute_sensors(theta):
    """
    Modelo bilateral de los receptores sensoriales.

    Definición de θ:
        θ = direction_to_light − heading     [normalizado a (−π, π]]

    Convención (fototaxia positiva):
        θ > 0  →  fuente a la IZQUIERDA  →  S_izq > S_der  →  error < 0  →  ω < 0
        θ < 0  →  fuente a la DERECHA    →  S_der > S_izq  →  error > 0  →  ω > 0
        θ = 0  →  agente apunta a la fuente (equilibrio)

    Modelo sensorial:
        S_der = 0.5 · (1 − sin θ)    # máx. cuando θ < 0 (fuente a la derecha)
        S_izq = 0.5 · (1 + sin θ)    # máx. cuando θ > 0 (fuente a la izquierda)

    Señal de error bilateral:
        error = S_der − S_izq = −sin θ. (diferencia de estimulación)
                  error > 0  →  fuente a la derecha  →  ω > 0 (giro derecho)
                  error < 0  →  fuente a la izquierda →  ω < 0 (giro izquierdo)

    Retorna: s_der, s_izq, error
    """
    baseline = 0.01  # sensibilidad mínima residual
    s_der = baseline + (1 - 2*baseline) * 0.5 * (1.0 - np.sin(theta))
    s_izq = baseline + (1 - 2*baseline) * 0.5 * (1.0 + np.sin(theta))
    return s_der, s_izq, s_der - s_izq


# ── Simulación ───────────────────────────────────────────────

def simulate_phototaxis(gain_k, v_forward, n_steps, theta0_deg,
                        light_pos, agent_start, dt=1.0,
                        blind_eye=None, noise_sigma=0.0):
    """
    Simula fototaxia con retroalimentación negativa (lazo cerrado).

    Ecuaciones del sistema:
    ──────────────────────
        θ     = normalize(dir_to_light − φ)    ángulo relativo agente–fuente
        error = S_der − S_izq = −sin θ         señal bilateral de error
        ω     = k · error                      tasa de giro
                ω > 0  →  fuente a la derecha   →  giro derecho
                ω < 0  →  fuente a la izquierda →  giro izquierdo
        φ'    = φ − ω · dt                     heading actualizado
        r'    = r + v · [cos φ', sin φ']       avance a velocidad constante

    El lazo se cierra porque el giro modifica φ → modifica θ → modifica error → modifica ω.
    Con k > 0 la retroalimentación es NEGATIVA: el error disminuye en cada paso.

    Parámetros
    ──────────
    gain_k     : float  — ganancia k (> 0: fototaxia positiva; < 0: negativa)
    v_forward  : float  — velocidad de avance (constante)
    n_steps    : int    — pasos máximos de simulación
    theta0_deg : float  — ángulo relativo inicial en grados
    light_pos  : array  — posición [x, y] de la fuente de luz
    agent_start: array  — posición [x, y] inicial del agente
    dt         : float  — tamaño del paso temporal
    blind_eye  : str    — None | 'left' | 'right'  (experimento ojo cubierto)

    Retorna
    ───────
    positions    : (n+1, 2) — trayectoria espacial
    headings     : (n+1,)   — heading en coords. mundo (rad)
    thetas       : (n+1,)   — ángulo relativo θ (rad)
    errors       : (n+1,)   — señal de error S_der − S_izq
    s_ders       : (n+1,)   — estimulación receptor derecho
    s_izqs       : (n+1,)   — estimulación receptor izquierdo
    converged_at : int      — paso en que el agente alcanza la fuente
    """
    pos     = agent_start.copy().astype(float)

    # Heading inicial: heading = dir_to_light − θ₀
    dir0    = np.arctan2(light_pos[1] - pos[1], light_pos[0] - pos[0])
    heading = normalize_angle(dir0 - np.deg2rad(theta0_deg))

    # Arrays de resultados
    positions = np.zeros((n_steps + 1, 2))
    headings  = np.zeros(n_steps + 1)
    thetas    = np.zeros(n_steps + 1)
    errors    = np.zeros(n_steps + 1)
    s_ders    = np.zeros(n_steps + 1)
    s_izqs    = np.zeros(n_steps + 1)
    converged_at = n_steps

    # Estado inicial (paso 0)
    positions[0] = pos.copy()
    headings[0]  = heading
    dtl_init     = np.arctan2(light_pos[1] - pos[1], light_pos[0] - pos[0])
    theta_init   = normalize_angle(dtl_init - heading)
    sd, si, err  = compute_sensors(theta_init)
    thetas[0]    = theta_init
    errors[0]    = err
    s_ders[0]    = sd
    s_izqs[0]    = si

    # ── Bucle de simulación ──
    for i in range(n_steps):

        # Condición de equilibrio: agente llega a la vecindad de la fuente
        if np.linalg.norm(light_pos - pos) < 0.2:
            converged_at    = i
            positions[i+1:] = pos
            headings[i+1:]  = heading
            thetas[i+1:]    = 0.0
            errors[i+1:]    = 0.0
            s_ders[i+1:]    = 0.5
            s_izqs[i+1:]    = 0.5
            break

        # 1. Ángulo relativo θ entre heading actual y dirección a la fuente
        dtl   = np.arctan2(light_pos[1] - pos[1], light_pos[0] - pos[0])
        theta = normalize_angle(dtl - heading)

        # 2. Señal sensorial bilateral
        s_der, s_izq, error = compute_sensors(theta)

        # 3. Modificación por oclusión de receptor (experimento ojo cubierto)
        if blind_eye == 'left':
            s_izq = 0.0
            error = s_der - s_izq   # señal solo desde receptor derecho → ω siempre > 0
        elif blind_eye == 'right':
            s_der = 0.0
            error = s_der - s_izq   # señal solo desde receptor izquierdo → ω siempre < 0

        # 4. Tasa de giro: ω = k · error
        #    ω > 0 → fuente a la derecha → girar derecha (heading disminuye)
        #    ω < 0 → fuente a la izquierda → girar izquierda (heading aumenta)
        omega = gain_k * error

        # 5. Actualización del heading y posición
        noise   = np.random.normal(0.0, noise_sigma) * dt if noise_sigma > 0 else 0.0
        heading = normalize_angle(heading - omega * dt + noise)
        pos     = pos + v_forward * np.array([np.cos(heading), np.sin(heading)])

        # 6. Registrar estado del paso i+1
        positions[i + 1] = pos.copy()
        headings[i + 1]  = heading
        thetas[i + 1]    = theta
        errors[i + 1]    = error
        s_ders[i + 1]    = s_der
        s_izqs[i + 1]    = s_izq

    return positions, headings, thetas, errors, s_ders, s_izqs, converged_at


# ── Gráficas ─────────────────────────────────────────────────

def _plot_celda1():
    (positions, headings, thetas, errors,
     s_ders, s_izqs, converged_at) = simulate_phototaxis(
        gain_k      = GAIN_K,
        v_forward   = V_FORWARD,
        n_steps     = N_STEPS,
        theta0_deg  = THETA0_DEG,
        light_pos   = LIGHT_POS,
        agent_start = AGENT_START,
        noise_sigma = NOISE_SIGMA
    )

    steps      = np.arange(N_STEPS + 1)
    thetas_deg = np.rad2deg(thetas)

    fig = plt.figure(figsize=(18, 8), facecolor=P['DARK_BG'])
    fig.suptitle(
        "Fototaxia positiva:  Comparación Bilateral Simultánea  (retroalimentación negativa)",
        color=P['TEXT'], fontsize=13, fontweight='bold', y=.98
    )
    fig.text(
        0.5, 0.93,
        f"  Parámetros:   k = {GAIN_K} | θ₀ = {THETA0_DEG}° | Pasos = {N_STEPS} | Ruido = {NOISE_SIGMA}",
        color=P['SUBTEXT'], fontsize=9, ha='center', va='top', fontstyle='italic'
    )

    gs   = fig.add_gridspec(1, 3, wspace=0.48)
    ax_A = fig.add_subplot(gs[0])
    ax_B = fig.add_subplot(gs[1])
    ax_C = fig.add_subplot(gs[2])

    for ax in [ax_A, ax_B, ax_C]:
        ax.set_facecolor(P['PANEL_BG'])
        ax.tick_params(colors=P['SUBTEXT'], labelsize=9)
        for spine in ax.spines.values():
            spine.set_edgecolor(P['LINE'])
        ax.grid(True, color=P['LINE'], linewidth=0.5, alpha=0.5)


    # ════════════════════════════════════════════════════════════
    # Panel A — Trayectoria espacial coloreada por S_der − S_izq
    # ════════════════════════════════════════════════════════════
    # Color de la trayectoria:
    #   Naranja (ACCENT1) → error > 0 → S_der > S_izq → fuente a la derecha
    #   Azul    (ACCENT2) → error < 0 → S_izq > S_der → fuente a la izquierda

    cmap_err = LinearSegmentedColormap.from_list(
        "error_cmap",
        [(0.0, P['ACCENT2']), (0.5, P['MIDTONE']), (1.0, P['ACCENT1'])],
        N=256
    )
    norm_err = Normalize(vmin=-1.0, vmax=1.0)

    n_traj   = max(converged_at, 1)
    traj_pos = positions[:n_traj + 1]
    traj_err = errors[:n_traj]
    pts      = traj_pos.reshape(-1, 1, 2)
    segs     = np.concatenate([pts[:-1], pts[1:]], axis=1)
    lc       = LineCollection(segs, cmap=cmap_err, norm=norm_err,
                               linewidth=2.2, alpha=0.93, zorder=3)
    lc.set_array(traj_err)
    ax_A.add_collection(lc)

    # Fuente de luz
    ax_A.add_patch(plt.Circle(LIGHT_POS, 1.3, color=P['GOLD'], alpha=0.09, zorder=2))
    ax_A.add_patch(plt.Circle(LIGHT_POS, 0.5, color=P['GOLD'], alpha=0.18, zorder=2))
    ax_A.scatter(*LIGHT_POS, s=300, color=P['GOLD'], zorder=6, marker='*',
                  edgecolors=P['TEXT'], linewidths=0.4, label='Fuente de luz')

    # Marcadores de inicio y convergencia
    ax_A.scatter(*positions[0], s=85, color=P['TEXT'], zorder=5,
                  marker='o', edgecolors=P['LINE'], linewidths=1.2, label='Inicio')
    ax_A.scatter(*positions[converged_at], s=85, color=P['ACCENT1'], zorder=5,
                  marker='D', edgecolors=P['TEXT'], linewidths=0.5, label='Convergencia')

    # Flechas de heading distribuidas a lo largo de la trayectoria
    n_arrows   = 12
    arrow_idxs = np.linspace(0, max(converged_at - 1, 1), n_arrows, dtype=int)
    for idx in arrow_idxs:
        px, py = positions[idx]
        hd     = headings[idx]
        ax_A.annotate('',
                       xy=(px + 0.85 * np.cos(hd), py + 0.85 * np.sin(hd)),
                       xytext=(px, py),
                       arrowprops=dict(arrowstyle='->', color=P['TEXT'],
                                       lw=1.0, alpha=0.5))

    # Colorbar
    sm   = plt.cm.ScalarMappable(cmap=cmap_err, norm=norm_err)
    cbar = fig.colorbar(sm, ax=ax_A, fraction=0.038, pad=0.03)
    cbar.set_label("Diferencia en la estimulación (S_d - S_i)", color=P['SUBTEXT'], fontsize=8)
    cbar.set_ticks([-1.0, 0.0, 1.0])
    cbar.ax.set_yticklabels(
        ["−1  (izq. dom.)", "  0  (centrado)", "+1  (der. dom.)"],
        color=P['SUBTEXT'], fontsize=7
    )
    cbar.ax.yaxis.set_tick_params(color=P['SUBTEXT'])
    cbar.ax.set_facecolor(P['PANEL_BG'])
    cbar.outline.set_edgecolor(P['LINE'])

    all_x  = np.concatenate([positions[:converged_at + 1, 0], [LIGHT_POS[0]]])
    all_y  = np.concatenate([positions[:converged_at + 1, 1], [LIGHT_POS[1]]])
    margin = 2.5
    ax_A.set_xlim(all_x.min() - margin, all_x.max() + margin)
    ax_A.set_ylim(all_y.min() - margin, all_y.max() + margin)
    ax_A.set_aspect('equal')
    ax_A.set_title("A — Trayectoria espacial", color=P['TEXT'], fontsize=11, pad=8)
    ax_A.set_xlabel("x", color=P['SUBTEXT'], fontsize=9)
    ax_A.set_ylabel("y", color=P['SUBTEXT'], fontsize=9)
    ax_A.legend(facecolor=P['PANEL_BG'], edgecolor=P['LINE'], labelcolor=P['TEXT'],
                 fontsize=8, loc='upper center',
                 bbox_to_anchor=(0.5, -0.14), ncol=3)


    # ════════════════════════════════════════════════════════════
    # Panel B — Orientación relativa θ(t)
    # Muestra cómo el ángulo relativo entre heading y fuente converge a 0°
    # ════════════════════════════════════════════════════════════

    ax_B.plot(steps, thetas_deg, color=P['GOLD'], linewidth=1.9, zorder=3)

    # Línea de equilibrio estable (θ = 0°)
    ax_B.axhline(0, color=P['ACCENT1'], linewidth=1.4, linestyle='--',
                  alpha=0.85, zorder=2, label='θ = 0°  (equilibrio estable)')

    # Relleno por dirección de la fuente:
    #   θ > 0 → fuente a izquierda → azul → ω < 0
    #   θ < 0 → fuente a derecha  → naranja → ω > 0
    ax_B.fill_between(steps, 0, thetas_deg, where=(thetas_deg > 0),
                       alpha=0.18, color=P['ACCENT2'],
                       label='θ > 0: fuente a izq.  →  ω < 0 (giro izq.)')
    ax_B.fill_between(steps, 0, thetas_deg, where=(thetas_deg < 0),
                       alpha=0.18, color=P['ACCENT1'],
                       label='θ < 0: fuente a der.  →  ω > 0 (giro der.)')

    # Anotación del ángulo inicial
    offset_y = 20 if thetas_deg[0] > 0 else -28
    ax_B.annotate(
        f"θ₀ = {THETA0_DEG:.0f}°",
        xy=(0, thetas_deg[0]),
        xytext=(N_STEPS * 0.07, thetas_deg[0] + offset_y),
        color=P['TEXT'], fontsize=8.5,
        arrowprops=dict(arrowstyle='->', color=P['SUBTEXT'], lw=0.9)
    )

    # Línea y anotación de convergencia
    if converged_at < N_STEPS:
        ax_B.axvline(converged_at, color=P['SUBTEXT'], linewidth=0.8,
                      linestyle=':', alpha=0.55)
        ax_B.annotate(
            f"paso {converged_at}",
            xy=(converged_at, -10),
            xytext=(converged_at + N_STEPS * 0.04, -40),
            color=P['SUBTEXT'], fontsize=7.5,
            arrowprops=dict(arrowstyle='->', color=P['SUBTEXT'], lw=0.7)
        )

    ax_B.set_title("B — Orientación relativa  θ(t)", color=P['TEXT'], fontsize=11, pad=8)
    ax_B.set_xlabel("Pasos", color=P['SUBTEXT'], fontsize=9)
    ax_B.set_ylabel("θ  (grados)", color=P['SUBTEXT'], fontsize=9)
    ax_B.set_xlim(0, N_STEPS)
    ax_B.set_ylim(-195, 195)
    ax_B.set_yticks([-180, -90, 0, 90, 180])
    ax_B.legend(facecolor=P['PANEL_BG'], edgecolor=P['LINE'], labelcolor=P['TEXT'],
                 fontsize=8, loc='upper center',
                 bbox_to_anchor=(0.5, -0.14), ncol=1)


    # ════════════════════════════════════════════════════════════
    # Panel C — Señal de error S_der − S_izq  vs. tiempo
    # Muestra cómo la diferencia bilateral evoluciona con la posición
    # ════════════════════════════════════════════════════════════

    ax_C.plot(steps, errors, color=P['GOLD'], linewidth=1.9, zorder=3)

    # Línea de error cero
    ax_C.axhline(0, color=P['TEXT'], linewidth=0.9, linestyle='--', alpha=0.4, zorder=2)

    # Relleno por signo del error
    ax_C.fill_between(steps, 0, errors, where=(errors >= 0),
                       alpha=0.28, color=P['ACCENT1'],
                       label='S_der > S_izq  →  fuente a la der.  →  ω > 0')
    ax_C.fill_between(steps, 0, errors, where=(errors < 0),
                       alpha=0.28, color=P['ACCENT2'],
                       label='S_izq > S_der  →  fuente a la izq.  →  ω < 0')

    # Anotación del error inicial
    offset_e = -0.18 if errors[0] > 0 else 0.12
    ax_C.annotate(
        f"err₀ = {errors[0]:.2f}",
        xy=(0, errors[0]),
        xytext=(N_STEPS * 0.07, errors[0] + offset_e),
        color=P['TEXT'], fontsize=8.5,
        arrowprops=dict(arrowstyle='->', color=P['SUBTEXT'], lw=0.9)
    )

    # Línea de convergencia
    if converged_at < N_STEPS:
        ax_C.axvline(converged_at, color=P['SUBTEXT'], linewidth=0.8,
                      linestyle=':', alpha=0.55)

    ax_C.set_title("C — Señal de error  S_der − S_izq", color=P['TEXT'], fontsize=11, pad=8)
    ax_C.set_xlabel("Pasos", color=P['SUBTEXT'], fontsize=9)
    ax_C.set_ylabel("S_der − S_izq  =  −sin(θ)", color=P['SUBTEXT'], fontsize=9)
    ax_C.set_xlim(0, N_STEPS)
    ax_C.set_ylim(-1.2, 1.2)
    ax_C.set_yticks([-1.0, -0.5, 0.0, 0.5, 1.0])
    ax_C.legend(facecolor=P['PANEL_BG'], edgecolor=P['LINE'], labelcolor=P['TEXT'],
                 fontsize=8, loc='upper center',
                 bbox_to_anchor=(0.5, -0.14), ncol=1)

    fig.subplots_adjust(top=0.87, bottom=0.32, left=0.06, right=0.96, wspace=0.48)
    plt.show()

    # ── Resumen numérico ──────────────────────────────────────
    print(f"\n{'─'*70}")
    print(f"  Parámetros:   k = {GAIN_K} | θ₀ = {THETA0_DEG}° | Pasos = {N_STEPS} | Ruido = {NOISE_SIGMA}")
    print(f"  Convergencia:      paso {converged_at} / {N_STEPS}")
    print(f"  θ en convergencia: {thetas_deg[converged_at]:.2f}°  (esperado ≈ 0°)")
    print(f"  error final:       {errors[converged_at]:.4f}  (esperado ≈ 0)")
    print(f"  Distancia final a la fuente:   "
          f"{np.linalg.norm(LIGHT_POS - positions[converged_at]):.3f} unidades")
    print(f"{'─'*70}")


# ── Toggle de tema + render ───────────────────────────────────
w_theme_s1 = widgets.ToggleButtons(
    options=[('Oscuro', 'dark'), ('Claro', 'light')],
    value='dark',
    description='',
    style={'button_width': '90px'},
    layout=widgets.Layout(margin='0 0 8px 0')
)
out_s1 = widgets.Output()

def _on_theme_s1(change):
    P.update(PALETTES[change['new']])
    with out_s1:
        clear_output(wait=True)
        plt.close('all')
        _plot_celda1()

w_theme_s1.observe(_on_theme_s1, names='value')

display(w_theme_s1, out_s1)
with out_s1:
    _plot_celda1()

ToggleButtons(layout=Layout(margin='0 0 8px 0'), options=(('Oscuro', 'dark'), ('Claro', 'light')), style=Toggl…

Output()

In [ ]:
#@title ─── *Celda 2: Simulador Interactivo Fototaxis* ────────────────────────
#@markdown **Ejercicios:**
#@markdown - **Básico:** Con $\theta_0 = 75°$ y $k = 1.0$, observa la trayectoria. ¿Es una línea recta o una espiral? Si doblas $k$, ¿cambia el equilibrio o solo la velocidad?
#@markdown - **Intermedio:** Activa "cubrir ojo izquierdo". ¿Qué ocurre? ¿Por qué el organismo gira continuamente?
#@markdown - **Avanzado:** Cambia el signo de $k$ (fototaxia negativa). ¿Hacia dónde se mueve ahora el organismo?

# ============================================================
# CELDA 2 — Simulador Interactivo de Fototaxia
# Retroalimentación Negativa · Comparación Bilateral Simultánea
# ============================================================

# Alias de conveniencia (siguen funcionando en el resto del código)
def _p(k): return P[k]

V_FWD = 1

# ══════════════════════════════════════════════════════════════
# FUNCIONES BASE  (2F)
# ══════════════════════════════════════════════════════════════

def _norm(a):
    return (a + np.pi) % (2 * np.pi) - np.pi

def _sensors(theta):
    baseline = 0.15
    s_der = baseline + (1 - 2*baseline) * 0.5 * (1.0 - np.sin(theta))
    s_izq = baseline + (1 - 2*baseline) * 0.5 * (1.0 + np.sin(theta))
    return s_der, s_izq, s_der - s_izq


# ══════════════════════════════════════════════════════════════
# SIMULACIÓN — UNA FUENTE
# ══════════════════════════════════════════════════════════════

def run_single(gain_k, n_steps, theta0_deg, light_pos, agent_start,
               dt=1.0, blind_eye=None, noise_sigma=0.0):
    pos     = agent_start.copy().astype(float)
    lp      = np.array(light_pos, dtype=float)
    heading = _norm(np.arctan2(lp[1]-pos[1], lp[0]-pos[0])
                    - np.deg2rad(theta0_deg))
    N = n_steps
    positions = np.zeros((N+1, 2));  headings = np.zeros(N+1)
    thetas    = np.zeros(N+1);       errors   = np.zeros(N+1)
    s_ders    = np.zeros(N+1);       s_izqs   = np.zeros(N+1)
    ca        = N
    positions[0] = pos.copy();  headings[0] = heading
    th0          = _norm(np.arctan2(lp[1]-pos[1], lp[0]-pos[0]) - heading)
    sd, si, err  = _sensors(th0)
    thetas[0]=th0; errors[0]=err; s_ders[0]=sd; s_izqs[0]=si

    for i in range(N):
        if np.linalg.norm(lp - pos) < 0.2:
            ca = i
            positions[i+1:]=pos;  headings[i+1:]=heading
            thetas[i+1:]=0;       errors[i+1:]=0
            s_ders[i+1:]=0.5;     s_izqs[i+1:]=0.5
            break
        theta         = _norm(np.arctan2(lp[1]-pos[1], lp[0]-pos[0]) - heading)
        sd, si, error = _sensors(theta)
        if blind_eye == 'left':
            si    = 0.0;  error = sd - si
        elif blind_eye == 'right':
            sd    = 0.0;  error = sd - si
        omega   = gain_k * error
        nz      = np.random.normal(0, noise_sigma)*dt if noise_sigma > 0 else 0.0
        heading = _norm(heading - omega*dt + nz)
        pos     = pos + V_FWD * np.array([np.cos(heading), np.sin(heading)])
        positions[i+1]=pos.copy();  headings[i+1]=heading
        thetas[i+1]=theta;          errors[i+1]=error
        s_ders[i+1]=sd;             s_izqs[i+1]=si

    return positions, headings, thetas, errors, s_ders, s_izqs, ca


# ══════════════════════════════════════════════════════════════
# SIMULACIÓN — DOS FUENTES
# ══════════════════════════════════════════════════════════════

def simulate_phototaxis_2lights(gain_k, n_steps, theta0_deg,
                                 light_pos1, light_pos2, agent_start,
                                 dt=1.0, blind_eye=None, noise_sigma=0.0):
    pos = agent_start.copy().astype(float)
    lp1 = np.array(light_pos1, dtype=float)
    lp2 = np.array(light_pos2, dtype=float)
    mid = (lp1 + lp2) / 2.0
    heading = _norm(np.arctan2(mid[1]-pos[1], mid[0]-pos[0])
                    - np.deg2rad(theta0_deg))
    N = n_steps
    positions = np.zeros((N+1, 2));  headings = np.zeros(N+1)
    thetas    = np.zeros(N+1);       errors   = np.zeros(N+1)
    s_ders    = np.zeros(N+1);       s_izqs   = np.zeros(N+1)
    ca        = N
    positions[0] = pos.copy();  headings[0] = heading

    for i in range(N):
        d1 = max(np.linalg.norm(lp1 - pos), 0.5)
        d2 = max(np.linalg.norm(lp2 - pos), 0.5)
        if min(d1, d2) < 0.2:
            ca = i
            positions[i+1:]=pos;  headings[i+1:]=heading
            thetas[i+1:]=0;       errors[i+1:]=0
            s_ders[i+1:]=0.5;     s_izqs[i+1:]=0.5
            break
        th1 = _norm(np.arctan2(lp1[1]-pos[1], lp1[0]-pos[0]) - heading)
        th2 = _norm(np.arctan2(lp2[1]-pos[1], lp2[0]-pos[0]) - heading)
        sd1, si1, _ = _sensors(th1)
        sd2, si2, _ = _sensors(th2)
        w1  = 1.0 / (d1 * d1);  w2 = 1.0 / (d2 * d2);  wt = w1 + w2
        sd    = (w1*sd1 + w2*sd2) / wt
        si    = (w1*si1 + w2*si2) / wt
        error = sd - si
        if blind_eye == 'left':
            si    = 0.0;  error = sd - si
        elif blind_eye == 'right':
            sd    = 0.0;  error = sd - si
        omega   = gain_k * error
        nz      = np.random.normal(0, noise_sigma)*dt if noise_sigma > 0 else 0.0
        heading = _norm(heading - omega*dt + nz)
        pos     = pos + V_FWD * np.array([np.cos(heading), np.sin(heading)])
        th_mid  = _norm(np.arctan2(mid[1]-pos[1], mid[0]-pos[0]) - heading)
        positions[i+1]=pos.copy();  headings[i+1]=heading
        thetas[i+1]=th_mid;         errors[i+1]=error
        s_ders[i+1]=sd;              s_izqs[i+1]=si

    return positions, headings, thetas, errors, s_ders, s_izqs, ca


# ══════════════════════════════════════════════════════════════
# VISUALIZACIÓN
# ══════════════════════════════════════════════════════════════

def _plot_panels(positions, headings, thetas, errors, s_ders, s_izqs,
                 converged_at, n_steps, gain_k, theta0_deg, noise_sigma,
                 light_sources, mode='single'):

    thetas_deg = np.rad2deg(thetas)
    steps      = np.arange(n_steps + 1)

    cmap_err = LinearSegmentedColormap.from_list(
        "err", [(0.0, P['ACCENT2']), (0.5, P['MIDTONE']), (1.0, P['ACCENT1'])], N=256)
    norm_err = Normalize(vmin=-1.0, vmax=1.0)

    fig = plt.figure(figsize=(18, 8), facecolor=P['DARK_BG'])

    mode_lbl = ("Dos fuentes  (potencia igual · ponderación 1/d²)"
                if mode == 'two_lights' else "Una fuente")
    fig.suptitle(
        f"Fototaxia · Comparación Bilateral Simultánea · {mode_lbl}",
        color=P['TEXT'], fontsize=13, fontweight='bold', y=0.98)
    fig.text(
        0.5, 0.93,
        f"k = {gain_k} | θ₀ = {theta0_deg:.0f}° | "
        f"Pasos = {n_steps} | Ruido σ = {noise_sigma:.3f}",
        color=P['SUBTEXT'], fontsize=9, ha='center', va='top', fontstyle='italic')

    gs   = fig.add_gridspec(1, 3, wspace=0.48)
    ax_A = fig.add_subplot(gs[0])
    ax_B = fig.add_subplot(gs[1])
    ax_C = fig.add_subplot(gs[2])

    for ax in [ax_A, ax_B, ax_C]:
        ax.set_facecolor(P['PANEL_BG'])
        ax.tick_params(colors=P['SUBTEXT'], labelsize=9)
        for sp in ax.spines.values():
            sp.set_edgecolor(P['LINE'])
        ax.grid(True, color=P['LINE'], linewidth=0.5, alpha=0.5)

    # ── Panel A ──────────────────────────────────────────────
    n_traj   = max(converged_at, 1)
    traj_pos = positions[:n_traj + 1]
    traj_err = errors[:n_traj]
    pts      = traj_pos.reshape(-1, 1, 2)
    segs     = np.concatenate([pts[:-1], pts[1:]], axis=1)
    lc       = LineCollection(segs, cmap=cmap_err, norm=norm_err,
                               linewidth=2.2, alpha=0.93, zorder=3)
    lc.set_array(traj_err)
    ax_A.add_collection(lc)

    for lp in light_sources:
        ax_A.add_patch(plt.Circle(lp, 1.3, color=P['GOLD'], alpha=0.09, zorder=2))
        ax_A.add_patch(plt.Circle(lp, 0.5, color=P['GOLD'], alpha=0.18, zorder=2))
        ax_A.scatter(*lp, s=300, color=P['GOLD'], zorder=6, marker='*',
                      edgecolors=P['TEXT'], linewidths=0.4)

    ax_A.scatter(*positions[0], s=85, color=P['TEXT'], zorder=5,
                  marker='o', edgecolors=P['LINE'], linewidths=1.2, label='Inicio')
    ax_A.scatter(*positions[converged_at], s=85, color=P['ACCENT1'], zorder=5,
                  marker='D', edgecolors=P['TEXT'], linewidths=0.5, label='Conv.')

    for idx in np.linspace(0, max(n_traj - 1, 1), 12, dtype=int):
        px, py = positions[idx];  hd = headings[idx]
        ax_A.annotate('',
            xy=(px + 0.85*np.cos(hd), py + 0.85*np.sin(hd)),
            xytext=(px, py),
            arrowprops=dict(arrowstyle='->', color=P['TEXT'], lw=1.0, alpha=0.5))

    sm = plt.cm.ScalarMappable(cmap=cmap_err, norm=norm_err)
    cb = fig.colorbar(sm, ax=ax_A, fraction=0.038, pad=0.03)
    cb.set_label("S_d − S_i", color=P['SUBTEXT'], fontsize=8)
    cb.set_ticks([-1.0, 0.0, 1.0])
    cb.ax.set_yticklabels(["−1 (izq.)", "  0", "+1 (der.)"],
                           color=P['SUBTEXT'], fontsize=7)
    cb.ax.yaxis.set_tick_params(color=P['SUBTEXT'])
    cb.ax.set_facecolor(P['PANEL_BG'])
    cb.outline.set_edgecolor(P['LINE'])

    all_x = list(positions[:n_traj+1, 0]) + [lp[0] for lp in light_sources]
    all_y = list(positions[:n_traj+1, 1]) + [lp[1] for lp in light_sources]
    mg    = 4.0
    ax_A.set_xlim(min(all_x) - mg, max(all_x) + mg)
    ax_A.set_ylim(min(all_y) - mg, max(all_y) + mg)
    ax_A.set_aspect('equal')
    ax_A.set_title("A — Trayectoria espacial", color=P['TEXT'], fontsize=11, pad=8)
    ax_A.set_xlabel("x", color=P['SUBTEXT'], fontsize=9)
    ax_A.set_ylabel("y", color=P['SUBTEXT'], fontsize=9)
    ax_A.legend(facecolor=P['PANEL_BG'], edgecolor=P['LINE'], labelcolor=P['TEXT'],
                 fontsize=8, loc='upper center', bbox_to_anchor=(0.5, -0.14), ncol=2)

    # ── Panel B ──────────────────────────────────────────────
    ax_B.plot(steps, thetas_deg, color=P['GOLD'], linewidth=1.9, zorder=3)
    ax_B.axhline(0, color=P['ACCENT1'], linewidth=1.4, linestyle='--',
                  alpha=0.85, zorder=2, label='θ = 0°  (equilibrio)')
    ax_B.fill_between(steps, 0, thetas_deg, where=(thetas_deg > 0),
                       alpha=0.18, color=P['ACCENT2'],
                       label='θ > 0: fuente a izq. → ω < 0')
    ax_B.fill_between(steps, 0, thetas_deg, where=(thetas_deg < 0),
                       alpha=0.18, color=P['ACCENT1'],
                       label='θ < 0: fuente a der. → ω > 0')

    if abs(thetas_deg[0]) > 5:
        offset_y = 20 if thetas_deg[0] > 0 else -28
        ax_B.annotate(f"θ₀ = {theta0_deg:.0f}°",
                       xy=(0, thetas_deg[0]),
                       xytext=(n_steps*0.07, thetas_deg[0] + offset_y),
                       color=P['TEXT'], fontsize=8.5,
                       arrowprops=dict(arrowstyle='->', color=P['SUBTEXT'], lw=0.9))

    if converged_at < n_steps:
        ax_B.axvline(converged_at, color=P['SUBTEXT'], linewidth=0.8,
                      linestyle=':', alpha=0.55)
        ax_B.annotate(f"paso {converged_at}",
                       xy=(converged_at, -10),
                       xytext=(converged_at + n_steps*0.04, -40),
                       color=P['SUBTEXT'], fontsize=7.5,
                       arrowprops=dict(arrowstyle='->', color=P['SUBTEXT'], lw=0.7))

    ax_B.set_title("B — Orientación relativa  θ(t)", color=P['TEXT'], fontsize=11, pad=8)
    ax_B.set_xlabel("Pasos", color=P['SUBTEXT'], fontsize=9)
    ax_B.set_ylabel("θ  (grados)", color=P['SUBTEXT'], fontsize=9)
    ax_B.set_xlim(0, n_steps);  ax_B.set_ylim(-195, 195)
    ax_B.set_yticks([-180, -90, 0, 90, 180])
    ax_B.legend(facecolor=P['PANEL_BG'], edgecolor=P['LINE'], labelcolor=P['TEXT'],
                 fontsize=8, loc='upper center', bbox_to_anchor=(0.5, -0.14), ncol=1)

    # ── Panel C ──────────────────────────────────────────────
    ax_C.plot(steps, errors, color=P['GOLD'], linewidth=1.9, zorder=3)
    ax_C.axhline(0, color=P['TEXT'], linewidth=0.9, linestyle='--',
                  alpha=0.4, zorder=2)
    ax_C.fill_between(steps, 0, errors, where=(errors >= 0),
                       alpha=0.28, color=P['ACCENT1'],
                       label='S_der > S_izq → fuente a der. → ω > 0')
    ax_C.fill_between(steps, 0, errors, where=(errors < 0),
                       alpha=0.28, color=P['ACCENT2'],
                       label='S_izq > S_der → fuente a izq. → ω < 0')

    if abs(errors[0]) > 0.05:
        offset_e = -0.18 if errors[0] > 0 else 0.12
        ax_C.annotate(f"err₀ = {errors[0]:.2f}",
                       xy=(0, errors[0]),
                       xytext=(n_steps*0.07, errors[0] + offset_e),
                       color=P['TEXT'], fontsize=8.5,
                       arrowprops=dict(arrowstyle='->', color=P['SUBTEXT'], lw=0.9))

    if converged_at < n_steps:
        ax_C.axvline(converged_at, color=P['SUBTEXT'], linewidth=0.8,
                      linestyle=':', alpha=0.55)

    ax_C.set_title("C — Señal de error  S_der − S_izq",
                    color=P['TEXT'], fontsize=11, pad=8)
    ax_C.set_xlabel("Pasos", color=P['SUBTEXT'], fontsize=9)
    ax_C.set_ylabel("S_der − S_izq", color=P['SUBTEXT'], fontsize=9)
    ax_C.set_xlim(0, n_steps);  ax_C.set_ylim(-1.2, 1.2)
    ax_C.set_yticks([-1.0, -0.5, 0.0, 0.5, 1.0])
    ax_C.legend(facecolor=P['PANEL_BG'], edgecolor=P['LINE'], labelcolor=P['TEXT'],
                 fontsize=8, loc='upper center', bbox_to_anchor=(0.5, -0.14), ncol=1)

    fig.subplots_adjust(top=0.87, bottom=0.32, left=0.06, right=0.96, wspace=0.48)
    return fig


# ══════════════════════════════════════════════════════════════
# PARÁMETROS POR DEFECTO Y PRESETS
# ══════════════════════════════════════════════════════════════

DEFAULTS = dict(
    gain_k=0.15,  theta0_deg=150.0,  n_steps=90,
    light_x=50.0, light_y=45.0,
    agent_x=0.0,  agent_y=0.0,
    noise_sigma=0.0,  blind_eye="Ambos ojos",
)

PRESETS_FOTOTAXIA = {
    "Fototaxia positiva  k = 1":
        dict(gain_k=1.0,  theta0_deg=168.0, n_steps=70,
             light_x=50,  light_y=45,  agent_x=0, agent_y=0,
             noise_sigma=0.0,  blind_eye="Ambos ojos",           mode='single'),
    "Fototaxia positiva, k doble;  k = 2":
        dict(gain_k=2.0,  theta0_deg=168.0, n_steps=70,
             light_x=50,  light_y=45,  agent_x=0, agent_y=0,
             noise_sigma=0.0,  blind_eye="Ambos ojos",           mode='single'),
    "Fototaxia positiva, k débil; k = 0.3":
        dict(gain_k=0.3,  theta0_deg=170.0, n_steps=90,
             light_x=50,  light_y=45,  agent_x=0, agent_y=0,
             noise_sigma=0.0,  blind_eye="Ambos ojos",           mode='single'),
    "Equilibrio inestable  θ = 180°":
        dict(gain_k=1.0,  theta0_deg=180.0, n_steps=75,
             light_x=50,  light_y=45,  agent_x=0, agent_y=0,
             noise_sigma=0.005, blind_eye="Ambos ojos",          mode='single'),
    "Cubrir ojo izquierdo":
        dict(gain_k=0.3,  theta0_deg=50.0, n_steps=200,
             light_x=50,  light_y=45,  agent_x=0, agent_y=0,
             noise_sigma=0.0,  blind_eye="Cubrir ojo izquierdo", mode='single'),
    "Cubrir ojo derecho":
        dict(gain_k=0.3,  theta0_deg=50.0, n_steps=200,
             light_x=50,  light_y=45,  agent_x=0, agent_y=0,
             noise_sigma=0.0,  blind_eye="Cubrir ojo derecho",   mode='single'),
    "Fototaxia negativa  k = −1":
        dict(gain_k=-1.0, theta0_deg=168.0, n_steps=70,
             light_x=50,  light_y=45,  agent_x=0, agent_y=0,
             noise_sigma=0.0,  blind_eye="Ambos ojos",           mode='single'),
    "Ruido ambiental":
        dict(gain_k=0.8,  theta0_deg=150.0, n_steps=80,
             light_x=50,  light_y=45,  agent_x=0, agent_y=0,
             noise_sigma=0.15, blind_eye="Ambos ojos",           mode='single'),
    "Dos fuentes de luz  ★":
        dict(gain_k=0.6,  theta0_deg=0.0,   n_steps=120,
             light_x=50,  light_y=70,  agent_x=0, agent_y=0,
             noise_sigma=0.01, blind_eye="Ambos ojos",           mode='two_lights'),
}


# ══════════════════════════════════════════════════════════════
# WIDGETS
# ══════════════════════════════════════════════════════════════

_state = {'mode': 'single'}

_sl  = {'description_width': '140px'}
_lay = widgets.Layout(width='360px')

w_k_i1     = widgets.FloatSlider(value=DEFAULTS['gain_k'],
                               min=-3.0,   max=3.0,   step=0.1,
                               description='Ganancia  k',
                               style=_sl, layout=_lay, continuous_update=False)
w_th0   = widgets.FloatSlider(value=DEFAULTS['theta0_deg'],
                               min=-180.0, max=180.0, step=1.0,
                               description='θ₀  (grados)',
                               style=_sl, layout=_lay, continuous_update=False)
w_steps = widgets.IntSlider  (value=DEFAULTS['n_steps'],
                               min=20,     max=200,   step=10,
                               description='Pasos',
                               style=_sl, layout=_lay, continuous_update=False)
w_lx    = widgets.FloatSlider(value=DEFAULTS['light_x'],
                               min=-80.0,  max=80.0,  step=1.0,
                               description='Fuente  x',
                               style=_sl, layout=_lay, continuous_update=False)
w_ly    = widgets.FloatSlider(value=DEFAULTS['light_y'],
                               min=-80.0,  max=80.0,  step=1.0,
                               description='Fuente  y',
                               style=_sl, layout=_lay, continuous_update=False)
w_ax    = widgets.FloatSlider(value=DEFAULTS['agent_x'],
                               min=-80.0,  max=80.0,  step=1,
                               description='Inicio agente  x',
                               style=_sl, layout=_lay, continuous_update=False)
w_ay    = widgets.FloatSlider(value=DEFAULTS['agent_y'],
                               min=-80.0,  max=80.0,  step=1,
                               description='Inicio agente  y',
                               style=_sl, layout=_lay, continuous_update=False)
w_noise = widgets.FloatSlider(value=DEFAULTS['noise_sigma'],
                               min=0.0,    max=1.0,   step=0.01,
                               description='Ruido  σ',
                               style=_sl, layout=_lay, continuous_update=False)
w_eye   = widgets.Dropdown(
    options=["Ambos ojos", "Cubrir ojo izquierdo", "Cubrir ojo derecho"],
    value=DEFAULTS['blind_eye'],
    description='Sensores',
    style=_sl,
    layout=widgets.Layout(width='360px')
)

btn_run_i1 = widgets.Button(
    description='▶  Correr simulación',
    button_style='success',
    layout=widgets.Layout(width='190px', height='34px')
)
btn_def_i1 = widgets.Button(
    description='↺  Parámetros por defecto',
    button_style='warning',
    layout=widgets.Layout(width='190px', height='34px')
)

# ── toggle de tema ──────────────────────────────────
w_theme_i1 = widgets.ToggleButtons(
    options=[('Oscuro', 'dark'), ('Claro', 'light')],
    value='dark',
    description='',
    style={'button_width': '90px'},
    layout=widgets.Layout(margin='6px 0 0 0')
)

out_i1 = widgets.Output()


# ══════════════════════════════════════════════════════════════
# CALLBACKS
# ══════════════════════════════════════════════════════════════

def _blind_eye_val():
    return {'Ambos ojos': None,
            'Cubrir ojo izquierdo': 'left',
            'Cubrir ojo derecho':   'right'}[w_eye.value]


def _do_run():
    gain_k    = w_k_i1.value
    n_steps   = w_steps.value
    theta0    = w_th0.value
    light_pos = np.array([w_lx.value, w_ly.value])
    agent_pos = np.array([w_ax.value, w_ay.value])
    noise     = w_noise.value
    blind_eye = _blind_eye_val()
    mode      = _state['mode']

    with out_i1:
        clear_output(wait=True)
        plt.close('all')

        if mode == 'two_lights':
            lp1 = np.array([ w_lx.value,  w_ly.value])
            lp2 = np.array([-w_lx.value,  w_ly.value])
            (positions, headings, thetas, errors,
             s_ders, s_izqs, ca) = simulate_phototaxis_2lights(
                gain_k, n_steps, theta0, lp1, lp2, agent_pos,
                blind_eye=blind_eye, noise_sigma=noise)
            light_sources = [lp1, lp2]
        else:
            (positions, headings, thetas, errors,
             s_ders, s_izqs, ca) = run_single(
                gain_k, n_steps, theta0, light_pos, agent_pos,
                blind_eye=blind_eye, noise_sigma=noise)
            light_sources = [light_pos]

        _plot_panels(
            positions, headings, thetas, errors, s_ders, s_izqs,
            ca, n_steps, gain_k, theta0, noise,
            light_sources, mode=mode)
        plt.show()

        td = np.rad2deg(thetas)
        print(f"\n{'─'*64}")
        print(f"  k = {gain_k}  |  θ₀ = {theta0:.0f}°  |"
              f"  Pasos = {n_steps}  |  σ = {noise:.3f}")
        print(f"  Sensores : {w_eye.value}")
        print(f"  Modo     : {'dos fuentes (1/d²)' if mode == 'two_lights' else 'una fuente'}")
        print(f"  Conv.    : paso {ca} / {n_steps}")
        print(f"  θ final  : {td[ca]:.2f}°   |   error final : {errors[ca]:.4f}")
        if mode != 'two_lights':
            print(f"  Dist. final a la fuente : "
                  f"{np.linalg.norm(light_pos - positions[ca]):.3f} unidades")
        print(f"{'─'*64}\n")


def _on_run(_):
    _do_run()


def _on_reset(_):
    _state['mode'] = 'single'
    w_k_i1.value     = DEFAULTS['gain_k']
    w_th0.value   = DEFAULTS['theta0_deg']
    w_steps.value = DEFAULTS['n_steps']
    w_lx.value    = DEFAULTS['light_x']
    w_ly.value    = DEFAULTS['light_y']
    w_ax.value    = DEFAULTS['agent_x']
    w_ay.value    = DEFAULTS['agent_y']
    w_noise.value = DEFAULTS['noise_sigma']
    w_eye.value   = DEFAULTS['blind_eye']


def _make_preset_cb(p):
    def _cb(_):
        _state['mode'] = p.get('mode', 'single')
        w_k_i1.value     = p['gain_k']
        w_th0.value   = p['theta0_deg']
        w_steps.value = p['n_steps']
        w_lx.value    = p['light_x']
        w_ly.value    = p['light_y']
        w_ax.value    = p['agent_x']
        w_ay.value    = p['agent_y']
        w_noise.value = p['noise_sigma']
        w_eye.value   = p['blind_eye']
        _do_run()
    return _cb


btn_run_i1.on_click(_on_run)
btn_def_i1.on_click(_on_reset)


def _on_theme_i1(change):
    P.update(PALETTES[change['new']])   # actualiza paleta global
    _rebuild_ui()                        # reconstruye etiquetas HTML
    _do_run()                            # regrafica con nuevos colores

w_theme_i1.observe(_on_theme_i1, names='value')


# ══════════════════════════════════════════════════════════════
# LAYOUT
# ══════════════════════════════════════════════════════════════

def _hdr(t):
    return widgets.HTML(
        f"<p style='color:{P['TEXT']};font-weight:bold;font-size:12px;"
        f"margin:6px 0 2px 2px'>{t}</p>")

def _sep():
    return widgets.HTML(
        f"<hr style='border:0;border-top:1px solid {P['LINE']};margin:4px 0'>")

def _note_2l():
    return widgets.HTML(
        f"<p style='color:{P['SUBTEXT']};font-size:10px;line-height:1.4;margin:6px 2px'>"
        f"<b style='color:{P['TEXT']}'>★ Modo dos fuentes</b><br>"
        f"Fuente 1 → (fuente x, fuente y)<br>"
        f"Fuente 2 → (−fuente x, fuente y)<br>"
        f"θ(t) se mide respecto al punto medio.<br>"
        f"La asimetría en 'inicio x' rompe la<br>"
        f"simetría vía ponderación 1/d².</p>")

_pbtn_lay = widgets.Layout(width='235px', height='30px')

ui_box = widgets.HBox(layout=widgets.Layout(
    gap='24px', align_items='flex-start', padding='8px'))


def _rebuild_ui():
    preset_btns = []
    for _name, _params in PRESETS_FOTOTAXIA.items():
        _bstyle = 'info' if _params.get('mode') == 'two_lights' else ''
        _b = widgets.Button(description=_name, button_style=_bstyle, layout=_pbtn_lay)
        _b.on_click(_make_preset_cb(_params))
        preset_btns.append(_b)

    controls = widgets.VBox([
        _hdr("Sistema"),
        w_k_i1, w_th0, w_steps,
        _sep(),
        _hdr("Posiciones"),
        w_lx, w_ly, w_ax, w_ay,
        _sep(),
        _hdr("Perturbaciones"),
        w_noise, w_eye,
        _sep(),
        widgets.HBox([btn_run_i1, btn_def_i1],
                      layout=widgets.Layout(gap='8px', margin='4px 0')),
        _sep(),
        _hdr("Tema"),
        w_theme_i1,
    ], layout=widgets.Layout(width='400px'))

    presets_col = widgets.VBox(
        [_hdr("Presets del capítulo"), *preset_btns, _note_2l()],
        layout=widgets.Layout(width='255px')
    )

    ui_box.children = [controls, presets_col]


# ── Render inicial ────────────────────────────────────────────
_rebuild_ui()
display(ui_box, out_i1)
_do_run()

Output()

---
## Simulador 2 — Efecto de la Demora

Un sistema de retroalimentación puede volverse inestable si hay una demora entre el momento en que se detecta el error y el momento en que se ejecuta la corrección.


In [ ]:
#@title ─── *Celda 3: Demora en la regulación del sistema de retroalimentación* ─────────

# ============================================================
# CELDA 4b — Simulador 2: Efecto de la demora
# Sistema de fototaxia con demora discreta pura

# Umbral teórico (aproximación lineal): D* = π / (2 · k · dt)

#   k=1.0, dt=0.05 → D* ≈ 31 pasos  ← inestabilidad fuera del rango visible
#   k=2.0, dt=0.05 → D* ≈ 16 pasos  ← visible dentro de T=20 s  ✓

# Demo 1: Progresión de delays con k=2.0 (tres regímenes)
# Demo 2: Mismo delay D=18, k=2.0 vs k=1.0 (interacción k × demora)
# ============================================================

# ── Parámetros fijos ──────────────────────────────────────────
DT       = 0.05    # paso temporal (s)
T_TOTAL  = 20.0    # duración de la simulación (s)
THETA0   = 75.0    # ángulo inicial (grados)
K_DEMO   = 2.0     # ganancia usada en Demo 1
                   # D* = π/(2·K_DEMO·DT) ≈ 15.7 pasos


# ══════════════════════════════════════════════════════════════
# NÚCLEO: SIMULACIÓN CON DEMORA DISCRETA
# ══════════════════════════════════════════════════════════════

def simular_con_demora_demo(theta0_deg, k=2.0, delay_steps=0,
                       dt=DT, T=T_TOTAL, clip_deg=310.0):
    """
    Simula el ángulo θ(t) de un sistema de fototaxia con demora discreta.

    Ecuación de actualización
    ─────────────────────────
        θ[n+1] = θ[n] − k · sin(θ[n−D]) · dt

    La señal de error es sin(θ):
      · Se anula exactamente en θ=0 (equilibrio estable)
      · Captura signo y magnitud sin linealizar
      · La demora D desacopla la corrección del estado actual

    Umbral de inestabilidad (aprox. lineal, |θ| pequeño):
        D* = π / (2 · k · dt)

    Parámetros
    ----------
    theta0_deg  : ángulo inicial en grados
    k           : ganancia del sistema
    delay_steps : pasos entre detección del error y ejecución de la corrección
    dt          : paso temporal (s)
    T           : duración total (s)
    clip_deg    : |θ| máximo antes de declarar divergencia y rellenar con NaN

    Retorna
    -------
    times  : array de tiempos
    thetas : array de θ en grados
    """
    theta    = np.radians(theta0_deg)
    buf_size = max(delay_steps, 1)
    buffer   = deque([0.0] * buf_size, maxlen=buf_size)

    thetas = [np.degrees(theta)]
    times  = [0.0]
    t      = 0.0

    while t < T:
        error = np.sin(theta)               # señal de error actual

        if delay_steps == 0:
            omega = k * error               # sin demora: corrección inmediata
        else:
            omega = buffer.popleft()        # corrección calculada hace D pasos
            buffer.append(k * error)        # encolar corrección actual

        theta  = theta - omega * dt
        t     += dt

        thetas.append(np.degrees(theta))
        times.append(t)

        # Divergencia: rellenar el resto con NaN para indicar inestabilidad
        if abs(np.degrees(theta)) > clip_deg:
            remaining = int((T - t) / dt)
            thetas.extend([np.nan] * remaining)
            times.extend((t + np.arange(1, remaining + 1) * dt).tolist())
            break

    return np.array(times), np.array(thetas)


# ══════════════════════════════════════════════════════════════
# DEMO 1 — Progresión: cuatro regímenes con k = K_DEMO
# ══════════════════════════════════════════════════════════════

def _plot_demo1(ax_list):
    """
    Cuatro subpaneles con escala Y independiente.
    Cada uno muestra un régimen distinto del sistema:
        D=0  → convergencia suave
        D=8  → oscilaciones amortiguadas
        D=15 → umbral crítico (D ≈ D*)
        D=20 → inestabilidad (diverge)
    """
    D_STAR = np.pi / (2 * K_DEMO * DT)   # umbral teórico ≈ 15.7

    configs = [
        dict(delay_steps=0,  k=K_DEMO, label="delay = 0   →  Convergencia suave",
             color=P['GREEN'], ylim=(-15,  90)),
        dict(delay_steps=8,  k=K_DEMO, label="delay = 8   →  Oscilaciones amortiguadas",
             color=P['ACCENT2'], ylim=(-90,  90)),
        dict(delay_steps=15, k=K_DEMO, label=f"delay = 15  →  Umbral crítico (D ≈ {D_STAR:.1f})",
             color=P['GOLD'], ylim=(-230, 230)),
        dict(delay_steps=20, k=K_DEMO, label="delay = 20  →  INESTABILIDAD (diverge)",
             color=P['RED'], ylim=(-315, 315)),
    ]

    for ax, cfg in zip(ax_list, configs):
        t, theta = simular_con_demora_demo(
            theta0_deg=THETA0, k=cfg['k'], delay_steps=cfg['delay_steps'],
            dt=DT, T=T_TOTAL
        )

        ax.set_facecolor(P['PANEL_BG'])
        ax.tick_params(colors=P['SUBTEXT'], labelsize=8)
        for sp in ax.spines.values():
            sp.set_edgecolor(P['LINE'])
        ax.grid(True, color=P['LINE'], linewidth=0.4, alpha=0.45)

        # Zona de "inestabilidad" (|θ| > 90°) sombreada sutilmente
        ax.axhspan(-315, -90, alpha=0.06, color=P['LINE'],  zorder=0)
        ax.axhspan(  90,  315, alpha=0.06, color=P['LINE'],  zorder=0)
        ax.axhspan( -90,   90, alpha=0.04, color=P['LINE'], zorder=0)

        ax.axhline(0, color=P['TEXT'], lw=0.8, ls='--', alpha=0.35, zorder=2,
                   label='θ = 0° (equilibrio)')
        ax.plot(t, theta, color=cfg['color'], lw=1.9, zorder=3)

        # Indicador del delay en el eje x
        delay_t = cfg['delay_steps'] * DT
        if delay_t > 0:
            ax.axvline(delay_t, color=cfg['color'], lw=0.7, ls=':', alpha=0.5)
            ax.text(delay_t + 0.2, cfg['ylim'][1] * 0.88,
                    f"D·dt={delay_t:.2f}s",
                    color=P['SUBTEXT'], fontsize=7)

        ax.set_title(cfg['label'], color=cfg['color'], fontsize=8.5, pad=5)
        ax.set_xlabel("Tiempo (s)", color=P['SUBTEXT'], fontsize=8)
        ax.set_ylabel("θ  (grados)", color=P['SUBTEXT'], fontsize=8)
        ax.set_xlim(0, T_TOTAL)
        ax.set_ylim(cfg['ylim'])
        ax.legend(fontsize=7, framealpha=0.2, labelcolor=P['TEXT'],
                  facecolor=P['PANEL_BG'], edgecolor=P['LINE'])


# ══════════════════════════════════════════════════════════════
# DEMO 2 — Interacción k × delay (mismo D, dos valores de k)
# ══════════════════════════════════════════════════════════════

def _plot_demo2(ax):
    """
    Un panel: D=18, k=2.0 vs k=1.0.

    D=18 supera el umbral para k=2.0 pero no para k=1.0:
        k=2.0: α·D = k·dt·D = 2.0·0.05·18 = 1.80 > π/2 ≈ 1.57  →  INESTABLE
        k=1.0: α·D = k·dt·D = 1.0·0.05·18 = 0.90 < π/2 ≈ 1.57  →  Converge

    Pregunta pedagógica: ¿Por qué el mismo delay produce comportamientos opuestos?
    Respuesta: porque lo que controla la estabilidad es α·D = k·dt·D, no D solo.
    """
    D_FIXED = 18
    configs = [
        dict(k=2.0, color='#EC8B27',
             alpha_D=2.0 * DT * D_FIXED,
             label=f"k = 2.0,  D = {D_FIXED}  → INESTABLE"),
        dict(k=1.0, color='#5D3FD3',
             alpha_D=1.0 * DT * D_FIXED,
             label=f"k = 1.0,  D = {D_FIXED}  → Converge"),
    ]

    ax.set_facecolor(P['PANEL_BG'])
    ax.tick_params(colors=P['SUBTEXT'], labelsize=9)
    for sp in ax.spines.values():
        sp.set_edgecolor(P['LINE'])
    ax.grid(True, color=P['LINE'], linewidth=0.4, alpha=0.45)

    ax.axhspan(-315, -90, alpha=0.06, color=P['LINE'],   zorder=0)
    ax.axhspan(  90,  315, alpha=0.06, color=P['LINE'],   zorder=0)
    ax.axhspan( -90,   90, alpha=0.04, color=P['LINE'], zorder=0)
    ax.axhline(0, color=P['TEXT'], lw=0.8, ls='--', alpha=0.35, zorder=2)

    for cfg in configs:
        t, theta = simular_con_demora_demo(
            theta0_deg=THETA0, k=cfg['k'], delay_steps=D_FIXED,
            dt=DT, T=T_TOTAL
        )
        ax.plot(t, theta, color=cfg['color'], lw=2.1, label=cfg['label'], zorder=3)

    # Anotación del concepto clave
    ax.annotate(
        "Lo que controla la estabilidad es\nel PRODUCTO  k · D, "
        "no la demora \nni la ganancia por sí solos.",
        xy=(8.5, 200), fontsize=8.5, color=P['GOLD'],
        bbox=dict(boxstyle="round,pad=0.45", facecolor=P['PANEL_BG'],
                  edgecolor=P['GOLD'], alpha=0.85)
    )

    # Línea vertical en el inicio del delay
    ax.axvline(D_FIXED * DT, color=P['SUBTEXT'], lw=0.8, ls=':', alpha=0.5)
    ax.text(D_FIXED * DT + 0.15, 260,
            f"Corrección\narranca\n(t = {D_FIXED*DT:.2f}s)",
            color=P['SUBTEXT'], fontsize=7.5)

    ax.set_title(f"Mismo delay (D = {D_FIXED}), diferente k",
                 color=P['TEXT'], fontsize=10, pad=8)
    ax.set_xlabel("Tiempo (s)", color=P['SUBTEXT'], fontsize=9)
    ax.set_ylabel("θ  (grados)", color=P['SUBTEXT'], fontsize=9)
    ax.set_xlim(0, T_TOTAL)
    ax.set_ylim(-315, 315)
    ax.set_yticks([-270, -180, -90, 0, 90, 180, 270])
    ax.legend(fontsize=8.5, framealpha=0.25, labelcolor=P['TEXT'],
              facecolor=P['PANEL_BG'], edgecolor=P['LINE'],
              loc='lower right')


# ══════════════════════════════════════════════════════════════
# RENDER PRINCIPAL
# ══════════════════════════════════════════════════════════════

def _render():
    D_STAR = np.pi / (2 * K_DEMO * DT)

    fig = plt.figure(figsize=(16, 10), facecolor=P['DARK_BG'])
    fig.suptitle(
        "Simulador 2 — Efecto de la demora en el sistema de retroalimentación",
        color=P['TEXT'], fontsize=13, fontweight='bold', y=0.99
    )
    fig.text(
        0.5, 0.955,
        f"θ₀ = {THETA0}° · dt = {DT} s ",
        #f"[k = {K_DEMO} → D* ≈ {D_STAR:.1f} pasos]", Ocultar por simplicidad
        color=P['SUBTEXT'], fontsize=9, ha='center', fontstyle='italic'
    )

    gs_outer = gridspec.GridSpec(2, 1, figure=fig,
                                 hspace=0.48,
                                 top=0.90, bottom=0.07,
                                 left=0.06, right=0.97)

    # ── Demo 1: 4 paneles en fila ──────────────────────────────
    gs_top = gridspec.GridSpecFromSubplotSpec(
        1, 4, subplot_spec=gs_outer[0], wspace=0.38
    )
    ax_top = fig.text(
        0.5, 0.955,  # ya cubierto por suptitle y subtitle
        '', ha='center'
    )
    # Título de sección Demo 1
    fig.text(0.03, 0.93,
             "Demo 1 — Progresión de delays  [k = 2.0 fijo]",
             color=P['ACCENT1'], fontsize=10, fontweight='bold')

    axes_demo1 = [fig.add_subplot(gs_top[0, i]) for i in range(4)]
    _plot_demo1(axes_demo1)

    # ── Demo 2: 1 panel ancho ──────────────────────────────────
    fig.text(0.03, 0.46,
             "Demo 2 - Interacción k × delay  [mismo D = 18, dos valores de k]",
             color=P['ACCENT2'], fontsize=10, fontweight='bold')
    ax_demo2 = fig.add_subplot(gs_outer[1])
    _plot_demo2(ax_demo2)

    plt.show()

    # ── Resumen numérico ──────────────────────────────────────
    print(f"\n{'─'*65}")
    print("  UMBRALES TEÓRICOS  D* = π / (2 · k · dt)")
    print(f"{'─'*65}")
    print(f"{'─'*65}")
    print(f"  Demo 1:  k = {K_DEMO}  →  D* ≈ {np.pi/(2*K_DEMO*DT):.1f} pasos")
    print(f"  Demo 2:  D = 18  →  inestable para k=2.0 (α·D=1.80), "
          f"estable para k=1.0 (α·D=0.90)")
    print(f"{'─'*65}\n")


# ── Toggle de tema + render (idéntico a Celda 1) ─────────────
w_theme_s2 = widgets.ToggleButtons(
    options=[('Oscuro', 'dark'), ('Claro', 'light')],
    value='dark',
    description='',
    style={'button_width': '90px'},
    layout=widgets.Layout(margin='0 0 8px 0')
)
out_s2 = widgets.Output()

def _on_theme_s2(change):
    P.update(PALETTES[change['new']])
    with out_s2:
        clear_output(wait=True)
        plt.close('all')
        _render()

w_theme_s2.observe(_on_theme_s2, names='value')

display(w_theme_s2, out_s2)
with out_s2:
    _render()

ToggleButtons(layout=Layout(margin='0 0 8px 0'), options=(('Oscuro', 'dark'), ('Claro', 'light')), style=Toggl…

Output()

In [ ]:
#@title ─── *Celda 4: Módulo interactivo de demora*  ─────────────────────────────────────
#@markdown **Ejercicios:**
#@markdown - **Básico:** Con $k = 1.0$ y sin demora, confirma la convergencia suave. Añade una demora de 10 pasos. ¿Qué ocurre?
#@markdown - **Avanzado:** Encuentra el valor de demora a partir del cual el sistema comienza a oscilar para $k = 1.0$. Luego reduce $k$ a la mitad. ¿El mismo valor de demora ahora produce oscilaciones? ¿Por qué la ganancia y la demora interactúan?

# ============================================================
# SIMULADOR 2 — Efecto de la Demora
# Sistema de fototaxia con demora discreta
#
# Requiere: numpy, matplotlib, ipywidgets (preinstalados en Colab)
# ============================================================



# ── Constantes de simulación ──────────────────────────────────
DT      = 0.05    # paso temporal (s)
T_TOTAL = 20.0    # duración total (s)
CLIP    = 310.0   # umbral de divergencia (grados)


# ══════════════════════════════════════════════════════════════
# NÚCLEO: SIMULACIÓN CON DEMORA DISCRETA
# ══════════════════════════════════════════════════════════════

def simular_con_demora(theta0_deg, k=1.0, delay_steps=0,
                       dt=DT, T=T_TOTAL, clip_deg=CLIP):
    """
    Simula θ(t) de un sistema de fototaxia con demora discreta.

    Ecuación de actualización:
        θ[n+1] = θ[n] − k · sin(θ[n−D]) · dt

    Umbral de inestabilidad (aprox. lineal, |θ| pequeño):
        D* = π / (2 · k · dt)
    """
    theta    = np.radians(theta0_deg)
    buf_size = max(delay_steps, 1)
    buffer   = deque([0.0] * buf_size, maxlen=buf_size)

    thetas = [np.degrees(theta)]
    times  = [0.0]
    t      = 0.0

    while t < T:
        error = np.sin(theta)

        if delay_steps == 0:
            omega = k * error
        else:
            omega = buffer.popleft()
            buffer.append(k * error)

        theta  = theta - omega * dt
        t     += dt

        thetas.append(np.degrees(theta))
        times.append(t)

        if abs(np.degrees(theta)) > clip_deg:
            remaining = int((T - t) / dt)
            thetas.extend([np.nan] * remaining)
            times.extend((t + np.arange(1, remaining + 1) * dt).tolist())
            break

    return np.array(times), np.array(thetas)


# ══════════════════════════════════════════════════════════════
# PRESETS_DEMORA
# ══════════════════════════════════════════════════════════════

PRESETS_DEMORA = {
    'Sin demora (convergencia suave)':      dict(theta0=75.0, k=1.0, delay=0),
    'Demora leve (oscilación leve)':        dict(theta0=75.0, k=1.0, delay=8),
    'Demora grande (oscilación visible)':   dict(theta0=75.0, k=1.0, delay=25),
    'Demora grande, k reducido':            dict(theta0=75.0, k=0.5, delay=25),
    'k alto + demora → inestabilidad':      dict(theta0=75.0, k=2.0, delay=20),
}

# ══════════════════════════════════════════════════════════════
# CLASIFICACIÓN DEL RESULTADO
# ══════════════════════════════════════════════════════════════

def clasificar(thetas):
    """Devuelve ('stable'|'damped'|'unstable'), color, texto."""
    valid = thetas[~np.isnan(thetas)]
    if len(valid) < len(thetas):          # hubo NaN → divergió
        return 'unstable', P['RED'],    '⚠  INESTABLE — diverge'
    tail = valid[-min(80, len(valid)):]
    max_abs = np.max(np.abs(tail))
    if max_abs < 2.0:
        return 'stable',   P['GREEN'],  '✓  CONVERGENCIA'
    return 'damped',   P['GOLD'],   '~  OSCILACIÓN'


# ══════════════════════════════════════════════════════════════
# RENDER PRINCIPAL
# ══════════════════════════════════════════════════════════════

def render(theta0, k, delay, tema):
    P.update(PALETTES[tema])

    t, theta = simular_con_demora(theta0_deg=theta0, k=k,
                                   delay_steps=delay, dt=DT, T=T_TOTAL)

    kind, curve_color, status_txt = clasificar(theta)

    d_star = np.pi / (2 * k * DT)

    # ── Límites del eje Y ──────────────────────────
    valid = theta[~np.isnan(theta)]
    y_min = np.nanmin(theta) if len(valid) else -100
    y_max = np.nanmax(theta) if len(valid) else  100
    pad   = max((y_max - y_min) * 0.12, 15)
    y_lo  = max(-(CLIP + 5), y_min - pad)
    y_hi  = min(  CLIP + 5,  y_max + pad)

    # ── Figura ────────────────────────────────────
    fig = plt.figure(figsize=(11, 5.5), facecolor=P['DARK_BG'])

    gs = gridspec.GridSpec(1, 1, figure=fig,
                           top=0.87, bottom=0.12,
                           left=0.08, right=0.97)
    ax = fig.add_subplot(gs[0, 0])
    ax.set_facecolor(P['PANEL_BG'])

    # Zona inestable sombreada
    ax.axhspan(-CLIP, -90,  alpha=0.05, color=P['LINE'], zorder=0)
    ax.axhspan(  90,  CLIP, alpha=0.05, color=P['LINE'], zorder=0)

    # Línea de equilibrio punteada
    ax.axhline(0, color=P['TEXT'], lw=1.0, ls='--', alpha=0.4,
               zorder=2, label='θ = 0°  (equilibrio)')

    # Línea de delay vertical
    if delay > 0:
        delay_t = delay * DT
        ax.axvline(delay_t, color=P['SUBTEXT'], lw=0.8, ls=':', alpha=0.55)
        ax.text(delay_t + 0.2, y_hi * 0.88,
                f'D·dt = {delay_t:.2f} s',
                color=P['SUBTEXT'], fontsize=8.5, va='top')

    # Curva principal
    ax.plot(t, theta, color=curve_color, lw=2.2, zorder=3,
            label=f'θ(t)  —  {status_txt}')

    # Estilo de ejes
    ax.tick_params(colors=P['SUBTEXT'], labelsize=9)
    for sp in ax.spines.values():
        sp.set_edgecolor(P['LINE'])
    ax.grid(True, color=P['LINE'], linewidth=0.4, alpha=0.5)

    ax.set_xlim(0, T_TOTAL)
    ax.set_ylim(y_lo, y_hi)
    ax.set_xlabel('Tiempo (s)', color=P['SUBTEXT'], fontsize=10)
    ax.set_ylabel('θ  (grados)', color=P['SUBTEXT'], fontsize=10)

    ax.legend(fontsize=9, framealpha=0.25, labelcolor=P['TEXT'],
              facecolor=P['PANEL_BG'], edgecolor=P['LINE'],
              loc='upper right')

    # ── Título principal ───────────────────────────
    regime_map = {
        'stable':   ('Convergencia suave',        P['GREEN']),
        'damped':   ('Oscilación',     P['GOLD']),
        'unstable': ('INESTABILIDAD — diverge',    P['RED']),
    }
    reg_txt, reg_col = regime_map[kind]

    fig.suptitle(
        f'Simulador 2 — Módulo interactivo de demora',
        color=P['TEXT'], fontsize=13, fontweight='bold', y=0.98
    )

    # Subtítulo con parámetros + umbral
    stability_note = (
        f'D* ≈ {d_star:.1f} psos  →  D={delay} '
        + ('< D*  ✓  estable' if delay <= d_star else '> D*  ⚠  inestable')
        if delay > 0 else f'D* ≈ {d_star:.1f} pasos  (sin demora)'
    )
    fig.text(
        0.5, 0.925,
        f'θ₀ = {theta0}°  ·  k = {k}  ·  D = {delay} pasos  ·  dt = {DT} s'
        f'          [{stability_note}]',
        color=P['SUBTEXT'], fontsize=8.8, ha='center', fontstyle='italic'
    )

    # Anotación de régimen (esquina superior izquierda del panel)
    ax.text(0.012, 0.97, reg_txt,
            transform=ax.transAxes, fontsize=10, fontweight='bold',
            color=reg_col, va='top', ha='left',
            bbox=dict(boxstyle='round,pad=0.35', facecolor=P['PANEL_BG'],
                      edgecolor=reg_col, alpha=0.85))

    plt.show()

    # ── Resumen numérico ───────────────────────────
    print(f"{'─'*60}")
    print(f"  θ₀ = {theta0}°  |  k = {k}  |  D = {delay} pasos  |  dt = {DT} s")
    if delay > 0:
        alpha_D = k * DT * delay
        print(f"  α·D = k·dt·D = {alpha_D:.3f}   "
              f"({'> π/2 → INESTABLE' if alpha_D > np.pi/2 else '< π/2 → estable'})")
    final_valid = theta[~np.isnan(theta)]
    if len(final_valid) < len(theta):
        print(f"  θ final: DIVERGE (|θ| > {CLIP}°)")
    else:
        print(f"  θ final ≈ {final_valid[-1]:.3f}°")
    print(f"{'─'*60}")


# ══════════════════════════════════════════════════════════════
# WIDGETS
# ══════════════════════════════════════════════════════════════

# ── Sliders ───────────────────────────────────────────────────
w_theta = widgets.FloatSlider(
    value=75.0, min=-170.0, max=170.0, step=1.0,
    description='θ₀ (°)',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='480px'),
    continuous_update=False,
)
w_k_i2 = widgets.FloatSlider(
    value=1.0, min=0.1, max=4.0, step=0.1,
    description='k (ganancia)',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='480px'),
    continuous_update=False,
)
w_delay = widgets.IntSlider(
    value=0, min=0, max=50, step=1,
    description='D (pasos)',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='480px'),
    continuous_update=False,
)

# ── Tema ──────────────────────────────────────────────────────
w_theme_i2 = widgets.ToggleButtons(
    options=[('Oscuro', 'dark'), ('Claro', 'light')],
    value='dark',
    description='Tema:',
    style={'button_width': '100px', 'description_width': '50px'},
)

# ── Presets ───────────────────────────────────────────────────
w_preset_i2 = widgets.Dropdown(
    options=['— Seleccionar preset —'] + list(PRESETS_DEMORA.keys()),
    value='— Seleccionar preset —',
    description='Preset:',
    style={'description_width': '60px'},
    layout=widgets.Layout(width='400px'),
)

# ── Botones ───────────────────────────────────────────────────
btn_run_i2   = widgets.Button(
    description='▶  Correr simulación',
    button_style='primary',
    layout=widgets.Layout(width='190px', height='36px'),
)
btn_reset_i2 = widgets.Button(
    description='↺  Resetear',
    button_style='',
    layout=widgets.Layout(width='130px', height='36px'),
)

# ── Output ────────────────────────────────────────────────────
out_i2 = widgets.Output()


# ── Callbacks ─────────────────────────────────────────────────

def on_preset_change_i2(change):
    key = change['new']
    if key in PRESETS_DEMORA:
        p = PRESETS_DEMORA[key]
        w_theta.value = p['theta0']
        w_k_i2.value     = p['k']
        w_delay.value = p['delay']
        _run_i2()

def _run_i2():
    with out_i2:
        clear_output(wait=True)
        plt.close('all')
        render(
            theta0=w_theta.value,
            k=w_k_i2.value,
            delay=w_delay.value,
            tema=w_theme_i2.value,
        )

def on_run_i2(_):
    w_preset_i2.value = '— Seleccionar preset —'
    _run_i2()

def on_reset_i2(_):
    w_theta.value  = 75.0
    w_k_i2.value      = 1.0
    w_delay.value  = 0
    w_preset_i2.value = '— Seleccionar preset —'
    with out_i2:
        clear_output(wait=True)

def on_theme_i2(change):
    _run_i2()

w_preset_i2.observe(on_preset_change_i2, names='value')
btn_run_i2.on_click(on_run_i2)
btn_reset_i2.on_click(on_reset_i2)
w_theme_i2.observe(on_theme_i2, names='value')


# ══════════════════════════════════════════════════════════════
# LAYOUT Y DISPLAY
# ══════════════════════════════════════════════════════════════

header = widgets.HTML("""
<div style="
    font-family: arial;
    font-size: 13px;
    color: #8b949e;
    padding: 10px 4px 6px 4px;
    border-bottom: 1px solid #30363d;
    margin-bottom: 10px;
">
    <b style="color:#e6edf3; font-size:20px;">Simulador 2 — Efecto del producto de ganancia (k) y demora (D)</b><br>
</div>
""")

controls = widgets.VBox([
    header,
    widgets.HBox([w_theme_i2]),
    widgets.HTML("<hr style='border-color:#30363d; margin:6px 0'>"),
    w_theta,
    w_k_i2,
    w_delay,
    widgets.HTML("<hr style='border-color:#30363d; margin:6px 0'>"),
    widgets.HBox([
        widgets.Label('Presets:', layout=widgets.Layout(width='60px')),
        w_preset_i2,
    ]),
    widgets.HTML("<div style='height:6px'></div>"),
    widgets.HBox([btn_run_i2, widgets.HTML("&nbsp;"), btn_reset_i2]),
], layout=widgets.Layout(padding='6px'))

display(controls, out_i2)
_run_i2()

Output()

---
## **Simulador 3 — Reactivo vs. Anticipatorio**

Dos conductores viajan en una ruta de 500 km con gasolineras distribuidas irregularmente.

- El **conductor reactivo** carga cuando el nivel del tanque cae por debajo de un umbral.
- El **conductor anticipatorio** aprende la distribución de gasolineras y carga *antes* de tramos largos.

In [ ]:
#@title ── *Celda 5: Conductor Reactivo vs. Anticipatorio* ─────────────────
#@markdown **Ejercicios:**
#@markdown - **Básico:** Con la distribución por defecto [40, 90, 250, 280, 450 km], ¿cuál conductor llega al final? ¿En qué kilómetro se queda varado el reactivo?
#@markdown - **Avanzado:** Cambia las gasolineras a [20, 40, 60, 460, 480] —muchas al inicio, casi ninguna al final. ¿Puede el conductor reactivo tener éxito con algún umbral diferente? ¿Cuál es el umbral mínimo que garantiza llegar?

# Requiere: numpy, matplotlib, ipywidgets
# Variables de paleta y librerías (PALETTES, np, plt) definidas en celdas anteriores.


# ═══════════════════════════════════════════════════════════════════════════════
# BLOQUE 1 — Lógica de simulación
# ═══════════════════════════════════════════════════════════════════════════════

def _simular(gasolineras_km, autonomia_km, reactivo, umbral_reactivo, paso_km=1):
    """
    Simula un conductor en una ruta de 500 km.

    Orden por iteración: mover → verificar gasolinera → verificar combustible.
    Este orden garantiza que llegar a una gasolinera con el tanque exactamente
    vacío se trate como recarga exitosa (la física lo permite).

    El umbral de varado es -1e-9 en lugar de 0 para tolerar el ruido de punto
    flotante que acumula la resta iterativa de 1/autonomia_km.

    Estrategia REACTIVA:
        Carga si el tanque está por debajo del umbral al pasar por una
        gasolinera. No requiere conocimiento de la ruta.

    Estrategia ANTICIPATORIA:
        Carga si el combustible actual no alcanza para llegar a la siguiente
        gasolinera con un margen de seguridad del 5%. Requiere conocer la
        distribución de gasolineras.
    """
    consumo = 1.0 / autonomia_km
    MARGEN  = 0.05                          # margen de seguridad del 5 %
    pos     = 0.0
    tanque  = 1.0                           # 1.0 = tanque lleno
    gas_s   = sorted(gasolineras_km)

    hist = {
        "pos":       [0.0],
        "tanque":    [1.0],
        "cargas":    [],                    # km donde cargó
        "decisiones": [],                   # razonamiento del anticipatorio
    }
    varado = False

    while pos < 500:

        # 1 — Avanzar un paso
        tanque -= consumo * paso_km
        pos    += paso_km

        # 2 — Verificar gasolinera en la nueva posición
        en_gas = any(abs(pos - g) <= paso_km / 2.0 for g in gas_s)
        if en_gas:
            if reactivo:
                # Reactivo: carga si el tanque está bajo el umbral
                if tanque < umbral_reactivo:
                    hist["cargas"].append(pos)
                    tanque = 1.0
            else:
                # Anticipatorio: carga si no alcanzará a la siguiente estación
                proximas = [g for g in gas_s if g > pos + paso_km / 2.0]
                km_sig   = proximas[0] if proximas else 500.0
                dist_sig = km_sig - pos
                necesito = dist_sig * consumo + MARGEN
                if tanque < necesito:
                    hist["decisiones"].append(dict(
                        km         = pos,
                        tanque_pct = round(tanque * 100, 2),
                        sig_km     = km_sig,
                        dist       = dist_sig,
                        necesito   = round(necesito * 100, 2),
                    ))
                    hist["cargas"].append(pos)
                    tanque = 1.0

        # 3 — Verificar combustible (después de posible recarga)
        # Tolerancia de punto flotante: -1e-9 en lugar de 0
        if tanque < -1e-9:
            varado = True
            pos   -= paso_km
            break

        hist["pos"].append(round(pos, 1))
        hist["tanque"].append(max(tanque, 0.0))

    hist["pos"]      = np.array(hist["pos"])
    hist["tanque"]   = np.array(hist["tanque"])
    hist["varado"]   = varado
    hist["km_final"] = pos
    return hist


def _brechas(gasolineras_km, autonomia_km):
    """
    Calcula los tramos entre puntos de carga (inicio, gasolineras, fin)
    y los clasifica por peligrosidad respecto a la autonomía.
    """
    pts = sorted(set(
        [0.0] + [g for g in gasolineras_km if 0 <= g <= 500] + [500.0]
    ))
    resultado = []
    for i in range(len(pts) - 1):
        d = pts[i + 1] - pts[i]
        resultado.append(dict(
            desde     = pts[i],
            hasta     = pts[i + 1],
            dist      = d,
            critica   = d > autonomia_km,
            peligrosa = d > autonomia_km * 0.80,
        ))
    return resultado


# ═══════════════════════════════════════════════════════════════════════════════
# BLOQUE 2 — Visualización
# ═══════════════════════════════════════════════════════════════════════════════

def _paleta(oscuro):
    """Devuelve el diccionario de colores según el tema."""
    src = PALETTES['dark'] if oscuro else PALETTES['light']
    return dict(
        bg      = src['DARK_BG'],
        panel   = src['PANEL_BG'],
        texto   = src['TEXT'],
        sub     = src['SUBTEXT'],
        grid    = src['LINE'],
        accent1 = src['ACCENT1'],
        accent2 = src['ACCENT2'],
        gold    = src['GOLD'],
        midtone = src['MIDTONE'],
        green   = src['GREEN'],
        red     = src['RED'],
        purple  = src['PURPLE'],
    )


# ── Colores semánticos (independientes del tema) ──────────────────────────────
_COL_SEGURA   = "#449944"
_COL_PELIGRO  = "#DD8833"
_COL_CRITICA  = "#CC3333"
_COL_VARADO   = "#FF4444"
_COL_OK       = "#44BB66"


def _panel_brechas(ax, brechas, gas_km, aut, c):
    """
    Panel superior: barra horizontal que colorea cada tramo por peligrosidad.
    Sustituye los círculos de emoji por patches de matplotlib en la leyenda.
    """
    ax.set_xlim(0, 500)
    ax.set_ylim(0, 1)
    ax.set_yticks([])

    for b in brechas:
        col = (_COL_CRITICA  if b["critica"]
               else _COL_PELIGRO if b["peligrosa"]
               else _COL_SEGURA)
        alp = 0.85 if b["critica"] else 0.75 if b["peligrosa"] else 0.65

        ax.barh(0.50, b["dist"], left=b["desde"],
                height=0.62, color=col, alpha=alp,
                edgecolor=c["bg"], linewidth=0.8)

        if b["dist"] >= 18:
            ax.text(b["desde"] + b["dist"] / 2, 0.50,
                    f'{b["dist"]:.0f} km',
                    ha="center", va="center",
                    fontsize=8, color="white", fontweight="bold")

    # Marcadores de gasolineras
    for g in gas_km:
        if 0 < g < 500:
            ax.axvline(g, color=c["gold"], lw=2.0, alpha=0.9, zorder=5)
            ax.plot(g, 0.93, marker="v", color=c["gold"],
                    markersize=7, zorder=6, clip_on=False,
                    linestyle="none")

    # Leyenda con Patch de matplotlib — fuera del eje, a la derecha
    leyenda = [
        Patch(facecolor=_COL_SEGURA,  alpha=0.65,
              label=f"Segura (< {aut * 0.80:.0f} km)"),
        Patch(facecolor=_COL_PELIGRO, alpha=0.75,
              label=f"Peligrosa ({aut * 0.80:.0f}-{aut} km)"),
        Patch(facecolor=_COL_CRITICA, alpha=0.85,
              label=f"Imposible (> {aut} km)"),
    ]
    ax.legend(handles=leyenda, fontsize=8.5, framealpha=0.5,
              labelcolor=c["texto"], facecolor=c["panel"],
              edgecolor=c["grid"],
              loc="upper left", bbox_to_anchor=(1.01, 1.0),
              borderaxespad=0,
              title="Tramos", title_fontsize=8.5)

    ax.set_title("Analisis de tramos", color=c["texto"], fontsize=9.5, pad=5)
    ax.set_xlabel("Posicion en la ruta (km)", color=c["sub"], fontsize=9)
    for sp in ax.spines.values():
        sp.set_edgecolor(c["grid"])
    ax.tick_params(colors=c["sub"], labelsize=8.5)


def _panel_conductor(ax, hist, label, color, umbral, gas_km, c):
    """
    Panel de nivel de tanque para un conductor.
    Usa Line2D proxies y marcadores de triángulo en lugar de emoji.
    """
    pct = hist["tanque"] * 100

    # Área y línea principal
    ax.fill_between(hist["pos"], pct, 0, color=color, alpha=0.18)
    ax.plot(hist["pos"], pct, color=color, lw=2.2, zorder=4)

    # Zona de peligro reactivo
    ax.axhspan(0, umbral * 100, color=_COL_VARADO, alpha=0.07, zorder=1)
    ax.axhline(umbral * 100, color=_COL_VARADO, lw=1.1, ls=":",
               alpha=0.80, zorder=3)

    # Líneas de gasolineras disponibles
    for g in gas_km:
        if 0 < g < 500:
            ax.axvline(g, color=c["gold"], lw=1.0, ls="--", alpha=0.55, zorder=2)  # ← corregido

    # Eventos de carga: línea + triángulo en la parte superior del panel
    for km_c in hist["cargas"]:
        ax.axvline(km_c, color=color, lw=1.8, alpha=0.55, zorder=3)
        ax.plot(km_c, 107, marker="v", color=color, markersize=9,
                zorder=5, clip_on=False, linestyle="none")

    # Estado final: texto con caja de color
    if hist["varado"]:
        kf    = hist["km_final"]
        txt_x = min(kf + 7, 487)
        ax.axvline(kf, color=_COL_VARADO, lw=2.5, zorder=6)
        ax.fill_betweenx([0, 108], kf, 500,
                         color=_COL_VARADO, alpha=0.07)
        ax.text(txt_x, 54, f"VARADO\n km {kf:.0f}",
                color=_COL_VARADO, fontsize=9.5, fontweight="bold",
                va="center",
                bbox=dict(boxstyle="round,pad=0.35",
                          facecolor=c["panel"],
                          edgecolor=_COL_VARADO, alpha=0.92))
        status = f"Varado en km {kf:.0f}"
    else:
        ax.text(493, 54, "Llego\n500 km",
                color=_COL_OK, fontsize=9.5, fontweight="bold",
                ha="right", va="center",
                bbox=dict(boxstyle="round,pad=0.35",
                          facecolor=c["panel"],
                          edgecolor=_COL_OK, alpha=0.92))
        status = "Llego (500 km)"

    n = len(hist["cargas"])
    ax.set_title(f"{label}   |   {status}   |   {n} carga(s)",
                 color=c["texto"], fontsize=11, pad=7)
    ax.set_ylabel("Nivel del tanque (%)", color=c["sub"], fontsize=9)
    ax.set_ylim(0, 112)
    ax.set_xlim(0, 500)
    ax.yaxis.set_major_formatter(
        plt.FuncFormatter(lambda v, _: f"{v:.0f}%"))
    ax.tick_params(colors=c["sub"], labelsize=9)
    ax.grid(axis="y", color=c["grid"], alpha=0.45, lw=0.6)

    # Leyenda con proxies Line2D — fuera del eje, a la derecha
    handles = [
        Line2D([0], [0], color=color, lw=2.2,
               label="Nivel del tanque"),
        Line2D([0], [0], color=_COL_VARADO, lw=1.1, ls=":",
               label=f"Umbral reactivo ({umbral*100:.0f}%)"),
        Line2D([0], [0], color=c["gold"], lw=1.0, ls="--",   # ← corregido
               label="Gasolinera disponible"),
        Line2D([0], [0], color=color, marker="v", markersize=9,
               lw=0, label="Carga realizada"),
    ]
    ax.legend(handles=handles, fontsize=8.5, framealpha=0.4,
              labelcolor=c["texto"], facecolor=c["panel"],
              edgecolor=c["grid"],
              loc="upper left", bbox_to_anchor=(1.01, 1.0),
              borderaxespad=0)


def _graficar(gas_km, aut, umbral, oscuro):
    """Genera la figura de tres paneles y el log textual."""
    c   = _paleta(oscuro)                   # ← única fuente de verdad de colores
    r   = _simular(gas_km, aut, True,  umbral)
    a   = _simular(gas_km, aut, False, umbral)
    brs = _brechas(gas_km, aut)

    # ── Layout: panel de tramos + dos paneles de conductores ─────────────────
    fig = plt.figure(figsize=(14, 10.5), facecolor=c["bg"])   # ← corregido
    gs  = GridSpec(3, 1, figure=fig,
                   height_ratios=[0.9, 3, 3], hspace=0.52)

    ax_g = fig.add_subplot(gs[0])
    ax_r = fig.add_subplot(gs[1])
    ax_a = fig.add_subplot(gs[2])

    for ax in (ax_g, ax_r, ax_a):
        ax.set_facecolor(c["panel"])
        for sp in ax.spines.values():
            sp.set_edgecolor(c["grid"])
        ax.tick_params(colors=c["sub"])

    _panel_brechas(ax_g, brs, gas_km, aut, c)
    _panel_conductor(ax_r, r, "CONDUCTOR REACTIVO",      c["accent1"], umbral, gas_km, c)  # ← corregido
    _panel_conductor(ax_a, a, "CONDUCTOR ANTICIPATORIO", c["accent2"], umbral, gas_km, c)  # ← corregido

    ax_r.set_xlabel("")
    ax_a.set_xlabel("Posicion en la ruta (km)", color=c["sub"], fontsize=10)

    gas_str = (", ".join(str(int(g)) for g in sorted(gas_km))
               if gas_km else "ninguna")
    fig.suptitle(
        f"Reactivo vs. Anticipatorio  |  "
        f"Gasolineras: [{gas_str}]  |  "
        f"Autonomia: {aut} km  |  "
        f"Umbral reactivo: {umbral*100:.0f}%",
        color=c["texto"], fontsize=11.5, y=1.005, fontweight="bold",
    )

    # Margen derecho reservado para las leyendas externas (18 % del ancho)
    fig.subplots_adjust(right=0.82, hspace=0.52, top=0.95, bottom=0.07)
    plt.show()

    # ── Log textual ───────────────────────────────────────────────────────────
    SEP = "-" * 64

    print(f"\n{SEP}")
    print("Razonamiento del conductor ANTICIPATORIO:")
    print(SEP)
    if a["decisiones"]:
        for d in a["decisiones"]:
            print(
                f"  km {d['km']:>5.0f}  |  "
                f"Tanque: {d['tanque_pct']:5.1f}%  |  "
                f"Proxima gasolinera: km {d['sig_km']:.0f} "
                f"({d['dist']:.0f} km, necesita {d['necesito']:.1f}%)  ->  CARGA"
            )
    elif not a["varado"]:
        print("  No fue necesario cargar: el tanque alcanzo para toda la ruta.")
    else:
        print("  No pudo tomar decision: se varó antes de alcanzar una gasolinera.")

    print(f"\n{SEP}")
    print("Resumen comparativo:")
    print(SEP)

    def _est(h):
        return ("500 km  [llego]" if not h["varado"]
                else f"km {h['km_final']:.0f}  [varado]")

    print(f"  {'':28s}  {'REACTIVO':>20s}  {'ANTICIPATORIO':>14s}")
    print(f"  {'Cargas realizadas':28s}  {len(r['cargas']):>20d}"
          f"  {len(a['cargas']):>14d}")
    print(f"  {'Llego hasta':28s}  {_est(r):>22s}  {_est(a):>16s}")

    criticas = [b for b in brs if b["critica"]]
    if criticas:
        print(f"\n[!] Brecha(s) de mas de {aut} km — imposibles con tanque lleno:")
        for b in criticas:
            print(f"    km {b['desde']:.0f} -> {b['hasta']:.0f}  "
                  f"({b['dist']:.0f} km)  ->  ninguna estrategia puede cruzarla.")
    print(f"{SEP}\n")


# ═══════════════════════════════════════════════════════════════════════════════
# BLOQUE 3 — Escenarios predefinidos
# ═══════════════════════════════════════════════════════════════════════════════

PRESETS_CHOFER = {
    "Distribucion por defecto (moderada)": dict(g="40, 90, 250, 280, 450", aut=100, umb=0.15),
    "Muchas al inicio, pocas al final":    dict(g="20, 40, 60, 460, 480",  aut=100, umb=0.15),
    "Distribuidas uniformemente":          dict(g="100, 200, 300, 400",    aut=100, umb=0.15),
    "Un solo tramo muy largo en medio":    dict(g="50, 100, 400, 450",     aut=100, umb=0.15),
    "Autonomia reducida (60 km)":          dict(g="40, 90, 250, 280, 450", aut=60,  umb=0.15),
    "Umbral reactivo alto (40%)":          dict(g="40, 90, 250, 280, 450", aut=100, umb=0.40),
    "Sin gasolineras intermedias":         dict(g="",                       aut=100, umb=0.15),
    "Personalizado":                       None,
}


# ═══════════════════════════════════════════════════════════════════════════════
# BLOQUE 4 — Widget interactivo
# ═══════════════════════════════════════════════════════════════════════════════

_W        = widgets.Layout
_updating = False          # flag para evitar cascada de callbacks entre widgets


# ── Controles ─────────────────────────────────────────────────────────────────

w_titulo = widgets.HTML(
    "<h3 style='margin:6px 0 2px; font-family:sans-serif;'>"
    "Simulador 3 — Reactivo vs. Anticipatorio</h3>"
    "<p style='margin:0 0 8px; color:#888; font-size:13px;'>"
    "Dos conductores en una ruta de 500 km con gasolineras distribuidas "
    "irregularmente. El conductor <b>reactivo</b> carga cuando el tanque baja "
    "del umbral. El <b>anticipatorio</b> calcula si el combustible alcanzara "
    "hasta la siguiente estacion y carga antes del tramo largo.</p>"
)

w_preset = widgets.Dropdown(
    options=list(PRESETS_CHOFER.keys()),
    value="Distribucion por defecto (moderada)",
    description="Escenario:",
    style={"description_width": "85px"},
    layout=_W(width="440px"),
)

w_gas = widgets.Text(
    value="40, 90, 250, 280, 450",
    description="Gasolineras (km):",
    placeholder="km separados por coma  |  dejar vacio = sin gasolineras",
    style={"description_width": "130px"},
    layout=_W(width="460px"),
)

w_aut = widgets.IntSlider(
    value=100, min=50, max=400, step=10,
    description="Autonomia (km):",
    style={"description_width": "115px"},
    layout=_W(width="420px"),
)

w_umb = widgets.FloatSlider(
    value=0.15, min=0.05, max=0.95, step=0.05,
    description="Umbral reactivo:",
    readout_format=".0%",
    style={"description_width": "115px"},
    layout=_W(width="420px"),
)

w_tema = widgets.ToggleButton(
    value=True,
    description="Oscuro",
    button_style="",
    layout=_W(width="100px", height="32px"),
)

w_btn = widgets.Button(
    description="Simular",
    button_style="primary",
    layout=_W(width="110px", height="36px"),
)

w_out = widgets.Output()


# ── Callbacks ─────────────────────────────────────────────────────────────────

def _on_preset(change):
    """Al seleccionar un preset, actualiza los tres controles a la vez."""
    global _updating
    if _updating:
        return
    p = PRESETS_CHOFER.get(change["new"])
    if p is None:
        return                    # "Personalizado" no actualiza nada
    _updating = True
    w_gas.value = p["g"]
    w_aut.value = p["aut"]
    w_umb.value = p["umb"]
    _updating = False

w_preset.observe(_on_preset, names="value")


def _on_tema(change):
    w_tema.description = "Oscuro" if change["new"] else "Claro"

w_tema.observe(_on_tema, names="value")


def _marcar_personalizado(_):
    """Cualquier edición manual cambia el selector a 'Personalizado'."""
    global _updating
    if not _updating:
        w_preset.value = "Personalizado"

w_gas.observe(_marcar_personalizado, names="value")
w_aut.observe(_marcar_personalizado, names="value")
w_umb.observe(_marcar_personalizado, names="value")


def _click(_):
    with w_out:
        clear_output(wait=True)
        try:
            raw = w_gas.value.strip()
            if raw:
                gas = sorted({
                    float(x.strip())
                    for x in raw.split(",") if x.strip()
                })
                fuera = [g for g in gas if not (0 <= g <= 500)]
                if fuera:
                    print(f"[!] Gasolineras fuera del rango 0-500 km: {fuera}")
                    return
            else:
                gas = []                  # lista vacía = sin gasolineras

            _graficar(gas, w_aut.value, w_umb.value, w_tema.value)

        except ValueError:
            print("[!] Formato incorrecto. Usa numeros separados por coma, "
                  "ej: 40, 90, 250, 280, 450")

w_btn.on_click(_click)


# ── Ensamblado y despliegue ───────────────────────────────────────────────────

ui = widgets.VBox([
    w_titulo,
    widgets.HBox(
        [w_preset, w_tema],
        layout=_W(align_items="center", gap="16px"),
    ),
    w_gas,
    w_aut,
    w_umb,
    widgets.HBox([w_btn]),
    w_out,
], layout=_W(padding="12px 0px"))

display(ui)
w_btn.click()          # ejecucion inicial con el primer preset

## Ejercicio de cálculo manual (Ejercicio 3 del capítulo)

In [ ]:
#@title ── Celda 6: Ejercicio de cálculo manual ──────────────────────────────────
# Ejercicio 3 del capítulo:
# ω = k·(S_der − S_izq) con k = 2
# Verifica tu cálculo manual con esta celda.

k = 2.0

# Instante 1
s_der_1, s_izq_1 = 8.0, 5.0
error_1 = s_der_1 - s_izq_1
omega_1 = k * error_1
print(f"Instante 1: S_der={s_der_1}, S_izq={s_izq_1}")
print(f"  Error = {error_1:.1f}  →  ω = {omega_1:.1f}  →  La fuente está a la DERECHA")
print()

# Instante 2 (después del giro)
s_der_2, s_izq_2 = 6.5, 6.5
error_2 = s_der_2 - s_izq_2
omega_2 = k * error_2
print(f"Instante 2: S_der={s_der_2}, S_izq={s_izq_2}")
print(f"  Error = {error_2:.1f}  →  ω = {omega_2:.1f}  →  Equilibrio alcanzado")
print()
print("Reflexión:")
print("  En el instante 2 el error es exactamente 0.")
print("  Esto significa que el organismo está apuntando directamente hacia la fuente.")
print("  ¿Qué pasaría si hubiera un pequeño ruido en los sensores en ese momento?")


Instante 1: S_der=8.0, S_izq=5.0
  Error = 3.0  →  ω = 6.0  →  La fuente está a la DERECHA

Instante 2: S_der=6.5, S_izq=6.5
  Error = 0.0  →  ω = 0.0  →  Equilibrio alcanzado

Reflexión:
  En el instante 2 el error es exactamente 0.
  Esto significa que el organismo está apuntando directamente hacia la fuente.
  ¿Qué pasaría si hubiera un pequeño ruido en los sensores en ese momento?


---
## Créditos y licencia

Este notebook es parte del proyecto:

> **Bouzas, A. (2026).** *Aprendizaje y Comportamiento Adaptable: Principios y Modelos.*
> Lab25, Facultad de Psicología, UNAM.
> https://www.bouzaslab25.com

Apoyo en la construcción del simulador: **Eduardo Sánchez.**

Código disponible en: **https://github.com/bouzaslab25/libro-aca**
Licencia: [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)
